In [ ]:
import sys
import os
import warnings
import time
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import missingno as msno
import xgboost as xgb
import optuna
from optuna.pruners import MedianPruner
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.neighbors import LocalOutlierFactor, KNeighborsRegressor
from sklearn.ensemble import IsolationForest, RandomForestRegressor, StackingRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler, MaxAbsScaler, MinMaxScaler
from sklearn.inspection import permutation_importance
from IPython.display import display, display_html
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import ElasticNetCV, ElasticNet, RidgeCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.svm import SVR

# Add the parent directory to the system path to access the 'src' module.
# Assuming this notebook resides in 'notebooks/', '..' resolves to the project root.
sys.path.append(os.path.abspath('..'))

# Import custom utilities from the modularised source code.
# Note: 'SklearnWrapper' is imported here to ensure the class definition is available
# in the global namespace, which is critical for proper object serialisation.
from src.model_utils import SklearnWrapper, HousePricePreprocessor, save_production_model, load_production_model

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda x: "%.3f" % x)
pd.set_option("display.width", 500)

data_path = "../data/train.csv"
test_path = "../data/test.csv"

def load_data(path):
    data = pd.read_csv(path)
    return data

df = load_data(data_path)
df.head()

In [ ]:
test_df = load_data(test_path)
test_df.head()

In [ ]:
def check_df(dataframe, head=5):
    print("###################### Shape ######################")
    print(dataframe.shape)
    print("\n###################### Info ######################")
    dataframe.info()
    print("\n###################### Types ######################")
    display(dataframe.dtypes.to_frame("Type"))
    print("\n##################### Head #####################")
    display(dataframe.head(head))
    print("\n###################### Tail ######################")
    display(dataframe.tail(head))
    print("\n###################### Describe ######################")
    display(dataframe.describe([0, 0.05, 0.50, 0.95, 0.99, 1]).T)
    print("\n###################### Unique Value Counts per Column ######################")
    display(dataframe[dataframe.columns].nunique())
    print("\n###################### NA ######################")
    display(dataframe.isnull().sum().to_frame("Missing Values"))

check_df(df)

In [ ]:
# ========================================
# MSSubClass: Convert to Meaningful Categories
# ========================================
# MSSubClass is nominal categorical encoded as numbers
# Converting to readable category names for better interpretability

mssubclass_map = {
    20: '1storey_1946+',
    30: '1storey_1945-',
    40: '1storey_unf_attic',
    45: '1.5storey_unf',
    50: '1.5storey_fin',
    60: '2storey_1946+',
    70: '2storey_1945-',
    75: '2.5storey_all_ages',
    80: 'split_multilevel',
    85: 'split_foyer',
    90: 'duplex_all_style_age',
    120: '1storey_PUD_1946+',
    150: '1.5storey_PUD_all',
    160: '2storey_PUD_1946+',
    180: 'PUD_multilevel',
    190: '2family_conversion'
}

# Apply mapping
df['MSSubClass'] = df['MSSubClass'].map(mssubclass_map)

# Verify conversion
print("MSSubClass dtype:", df['MSSubClass'].dtype)
print("MSSubClass unique values:", df['MSSubClass'].nunique())
print("\nValue counts:")
print(df['MSSubClass'].value_counts())

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    """
    Returns the names of categorical, numerical, and categorical but cardinal columns in the dataset.
    
    Note: Categorical columns also include those that are numerically-coded but categorical in nature.
    
    Parameters
    ----------
        dataframe: pandas.DataFrame
                The dataframe from which the column names are to be extracted.
        cat_th: int, optional
                The threshold for the number of unique values to classify a numerical column as categorical.
        car_th: int, optional
                The threshold for the number of unique values to classify a categorical column as cardinal.
    
    Returns
    -------
        cat_cols: list
                A list of the names of categorical columns.
        num_cols: list
                A list of the names of numerical columns.
        cat_but_car: list
                A list of the names of categorical columns that are considered cardinal.
        num_but_cat: list
                A list of the names of numerical columns that are considered categorical.
    
    Examples
    --------
        import seaborn as sns
        df = sns.load_dataset("iris")
        print(grab_col_names(df))
    
    Notes
    -----
        - The sum of the lengths of cat_cols, num_cols, and cat_but_car equals the total number of columns.
        - Columns identified as "numeric but categorical" are included within the cat_cols list.
        - The total number of columns is equal to the sum of the lengths of the three returned lists: len(cat_cols) + len(num_cols) + len(cat_but_car).

    """
    
    # cat_cols, cat_but_car
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() <= cat_th and dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and dataframe[col].dtypes == "O"]

    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f"cat_cols: {len(cat_cols)}")
    print(f"num_cols: {len(num_cols)}")
    print(f"cat_but_car: {len(cat_but_car)}")
    print(f"num_but_cat: {len(num_but_cat)}")
    return cat_cols, num_cols, cat_but_car, num_but_cat

cat_cols, num_cols, cat_but_car, num_but_cat = grab_col_names(df)

num_cols = [col for col in num_cols if col not in "Id"]

In [ ]:
def cat_summary(dataframe, col_name, plot=False, ax=None):
    if dataframe[col_name].dtypes == "bool":
        dataframe[col_name] = dataframe[col_name].astype(int)
    value_counts = dataframe[col_name].value_counts(dropna=False)
    ratio = 100 * value_counts / len(dataframe)
    summary_df = pd.DataFrame({
        col_name: value_counts,
        "Ratio:%": ratio.round(2)
    })
    n_missing = dataframe[col_name].isnull().sum()
    #print(f"\n{'#'*50}")
    #print(f"Column: {col_name}")
    #print(f"{'#'*50}")
    #print(summary_df,"\n")
    if n_missing > 0:
        #print(f"Column: {col_name} has missing rate: {n_missing} ({100*n_missing/len(dataframe):.2f}%)\n")
        pass
    if plot:
        plot_data = dataframe[[col_name]].copy()
        plot_data[col_name] = plot_data[col_name].fillna("None")
        order = plot_data[col_name].value_counts().index
        
        if ax is None:
            plt.figure(figsize=(12, 6))
            ax = plt.gca()
        
        sns.countplot(x=col_name, data=plot_data, order=order, hue=col_name, palette="viridis", legend=False, ax=ax)
        total = len(plot_data)
        for p in ax.patches:
            height = p.get_height()
            pct = 100 * height / total
            ax.annotate(f"{int(height)}\n({pct: .2f}%)",
                       (p.get_x() + p.get_width() / 2., height + total * 0.01),
                       ha="center", va="bottom", fontsize=10, fontweight="bold")
        ax.set_title(f"{col_name} Distribution (Total n={total})", fontsize=14, fontweight="bold")
        ax.set_xlabel(col_name, fontsize=12)
        ax.set_ylabel("Count", fontsize=12)
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
        
        if ax is None:
            plt.tight_layout()
            plt.show(block=True)

# Display all graphs in a single plot
def plot_all_categories(dataframe, cat_cols, cols_per_row=4, save=False, filename="cat_summary.png"):
    n_cols = len(cat_cols)
    n_rows = (n_cols + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(n_rows, cols_per_row, figsize=(5*cols_per_row, 4*n_rows))
    axes = axes.flatten()
    
    for idx, col in enumerate(cat_cols):
        cat_summary(dataframe, col, plot=True, ax=axes[idx])
    
    for idx in range(n_cols, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    
    if save:
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        print(f"✓ Plot saved: {filename}")
    
    plt.show()

plot_all_categories(df, cat_cols)  # Don't save
# plot_all_categories(df, cat_cols, save=True)  # Save with default name
# plot_all_categories(df, cat_cols, save=True, filename="analyse.png")  # Save with special name

In [ ]:
def num_summary(dataframe, numerical_col, plot=False, ax=None):
    missing_count = dataframe[numerical_col].isna().sum()
    missing_pct = 100 * missing_count / len(dataframe)
    #print("#######################################################")
    #print(f"Column: {numerical_col} \nMissing Value Count: {missing_count} \nMissing Value Percentage: {missing_pct:.2f}%\n")
    #quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
    #print(dataframe[numerical_col].describe(quantiles).T, "\n")
    
    if plot:
        if ax is None:
            plt.figure(figsize=(10, 6))
            ax = plt.gca()
        
        dataframe[numerical_col].hist(ax=ax, bins=30, edgecolor="black")
        ax.set_xlabel(numerical_col, fontsize=12)
        ax.set_ylabel("Frequency", fontsize=12)
        ax.set_title(f"{numerical_col} Distribution", fontsize=14, fontweight="bold")
        ax.grid(alpha=0.3)
        
        if ax is None:
            plt.tight_layout()
            plt.show(block=True)

# Display all graphs in a single plot
def plot_all_numerical(dataframe, num_cols, cols_per_row=4, save=False, filename="num_summary.png"):
    n_cols = len(num_cols)
    n_rows = (n_cols + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(n_rows, cols_per_row, figsize=(5*cols_per_row, 4*n_rows))
    axes = axes.flatten()
    
    for idx, col in enumerate(num_cols):
        num_summary(dataframe, col, plot=True, ax=axes[idx])
    
    # Hide unused subplots
    for idx in range(n_cols, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    
    if save:
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        print(f"The graph has been saved: {filename}")
    
    plt.show()

plot_all_numerical(df, num_cols)  # Don't save
# plot_all_numerical(df, num_cols, save=True)  # Save with default name
# plot_all_numerical(df, num_cols, cols_per_row=3, save=True, filename="numerical_analysis.png")  # Save with special name

In [ ]:
def target_summary_with_cat(dataframe, target, categorical_col):
    """
    Displays the target mean for each class of the categorical variable (including NaN).
    
    Parameters
    ----------
    dataframe : pd.DataFrame
        The dataframe to analyse the relationship between the categorical feature and the target feature.
    target : str
        Target feature ('SalePrice')
    categorical_col : str
        Categorical feature
    
    Examples
    --------
    target_summary_with_cat(df, "SalePrice", "PoolQC")
    """
    # Temporarily fill NaNs with ‘None’ (without altering the original data)
    temp_cat_col = dataframe[categorical_col].fillna("None")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = pd.DataFrame({
            "COUNT": temp_cat_col.value_counts(),
            "RATIO": 100 * temp_cat_col.value_counts() / len(dataframe),
            "TARGET_MEAN": dataframe.groupby(temp_cat_col)[target].mean(),
            "TARGET_MEDIAN": dataframe.groupby(temp_cat_col)[target].median()
        }).sort_values("TARGET_MEAN", ascending=False)
    print(f"\n{'='*70}")
    print(f"{categorical_col} → {target}")
    print(f"{'='*70}")
    print(result, "\n")

# For categorical variables (NaN values will appear as None)
for col in cat_cols:
    target_summary_with_cat(df, "SalePrice", col)

In [ ]:
# For cardinal categorical variables (NaN values will appear as None)
for col in cat_but_car:
    target_summary_with_cat(df, "SalePrice", col)

In [ ]:
def target_summary_with_num(dataframe, target, numerical_col):
    """
    It displays the numerical variable statistics for the target variable across different groups.
    
    Parameters
    ----------
    dataframe : pd.DataFrame
        The dataframe to analyse the relationship between the numerical feature and the target feature.
    target : str
        Target variable ('SalePrice')
    numerical_col : str
        Numerical variable
    
    Examples
    --------
    target_summary_with_num(df, "SalePrice", "OverallQual")
    """
    # Categorise the target (divide into quartiles)
    target_bins = pd.qcut(dataframe[target], q=4, labels=["Q1_Low", "Q2_Medium", "Q3_High", "Q4_VeryHigh"])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = dataframe.groupby(target_bins)[numerical_col].agg([
            ('MEAN', 'mean'),
            ('MEDIAN', 'median'),
            ('STD', 'std'),
            ('MIN', 'min'),
            ('MAX', 'max')
        ]).round(2)
    
    print(f"\n{'='*70}")
    print(f"{numerical_col} by {target} Quartiles")
    print(f"{'='*70}")
    print(result)
    print()

# For numerical variables
for col in num_cols:
    target_summary_with_num(df, "SalePrice", col)

In [ ]:
def plot_correlation_heatmap(dataframe, 
                             columns=None, 
                             figsize=(20, 20), 
                             cmap="RdBu", 
                             annot_format=".2f", 
                             save=False, 
                             filename="heatmap.png", 
                             title=None):
    """
    Plots a correlation heatmap for numerical features.
    
    Parameters:
    -----------
    dataframe : pd.DataFrame
        Input dataframe
    columns : list, optional
        List of columns to include. If None, uses all numeric columns
    figsize : tuple, default (20, 20)
        Figure size (width, height)
    cmap : str, default "RdBu"
        Colourmap for heatmap (e.g., "RdBu", "coolwarm", "viridis")
    annot_format : str, default ".2f"
        Format for annotation values (e.g., ".2f", ".3f", ".1%")
    save : bool, default False
        Whether to save the figure
    filename : str, default "heatmap.png"
        Filename for saved figure (only used if save=True)
    title : str, optional
        Title for the plot. If None, uses default title
    
    Returns:
    --------
    tuple : (fig, ax, corr_matrix)
        Figure, axes, and correlation matrix
    """

    if columns is None:
        # Auto-detect numeric columns using grab_col_names
        _, numeric, _, numeric_but_cat = grab_col_names(dataframe)

        # Remove Id column
        numeric = [col for col in numeric if col != "Id"]

        # Include numeric but categorical columns
        columns = numeric + numeric_but_cat

    # Calculate correlation matrix
    corr_matrix = dataframe[columns].corr()

    # Plot
    fig, ax, = plt.subplots(figsize=figsize)

    with sns.axes_style("darkgrid"):
        sns.heatmap(corr_matrix,
                    annot=True,
                    fmt=annot_format,
                    cmap=cmap,
                    square=True,
                    linewidths=0.5,
                    cbar_kws={"shrink": 0.8},
                    ax=ax
                    )
    
    if title is None:
        title = "Correlation Heatmap"
    ax.set_title(title, fontsize=16, fontweight="bold", pad=20)

    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()

    if save:
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        print(f"The heatmap has been saved: {filename}")

    plt.show()

    return fig, ax, corr_matrix

plot_correlation_heatmap(df)

In [ ]:
_cramers_v_cache = {}

def cramers_v(x, y):
    """
    Calculate Cramér's V statistic for categorical-categorical association.
    Calculate Cramér's V statistic with manual caching.

    Cramér's V is a measure of association between two nominal variables,
    giving a value between 0 and 1 (inclusive).
    - 0.0-0.1: No or very weak association
    - 0.1-0.3: Weak association
    - 0.3-0.5: Moderate association
    - 0.5-0.7: Strong association
    - 0.7-1.0: Very strong association
    
    Parameters
    ----------
    x : pd.Series
        First categorical variable
    y : pd.Series
        Second categorical variable
    
    Returns
    -------
    float
        Cramér's V value (0-1)
    
    Examples
    --------
        cramers_v(df['MSZoning'], df['Neighborhood'])
    
    Notes
    -----
        Uses bias correction for better accuracy with small samples.
    """
    # Create cache key: use column names (memory efficient)
    cache_key = (x.name, y.name) if hasattr(x, 'name') and hasattr(y, 'name') else None
    
    # If in cache, return directly
    if cache_key and cache_key in _cramers_v_cache:
        return _cramers_v_cache[cache_key]
    
    # Otherwise calculate
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    result = np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))
    
    # Cache the result
    if cache_key:
        _cramers_v_cache[cache_key] = result
        # Also cache the reverse key (symmetric matrix)
        _cramers_v_cache[(y.name, x.name)] = result
    
    return result

def categorical_correlation_matrix(dataframe, cat_cols):
    """
    Create a correlation matrix for categorical variables using Cramér's V.
    
    Parameters
    ----------
    dataframe : pd.DataFrame
        The dataframe containing categorical variables
    cat_cols : list
        List of categorical column names
    
    Returns
    -------
    pd.DataFrame
        Correlation matrix with Cramér's V values (0-1)
    
    Examples
    --------
        cat_corr = categorical_correlation_matrix(df, cat_cols)
        display(cat_corr)
    
    Notes
    -----
        - NaN values are temporarily filled with "None" for calculation
        - Returns a symmetric matrix (V(X,Y) = V(Y,X))
        - Diagonal values are 1.0 (perfect correlation with itself)
    """
    # Temporarily fill NaN values with "None"
    df_temp = dataframe[cat_cols].fillna("None")

    # Initialize correlation matrix
    n = len(cat_cols)
    corr_matrix = pd.DataFrame(np.zeros((n, n)), index=cat_cols, columns=cat_cols)
    print("="*70)
    print("CALCULATING CRAMÉR'S V CORRELATION MATRIX")
    print("="*70)
    print(f"Number of categorical variables: {n}")
    print(f"Total pairs to calculate: {n*(n-1)//2}\n")

    # Calculate Cramer's V for each pair
    for i, col1 in enumerate(cat_cols):
        for j, col2 in enumerate(cat_cols):
            if i == j:
                corr_matrix.loc[col1, col2] = 1.0 # Perfect correlation with itself
            elif i < j: # Calculate only upper triangle (symmetric matrix)
                try:
                    v = cramers_v(df_temp[col1], df_temp[col2])
                    corr_matrix.loc[col1, col2] = v
                    corr_matrix.loc[col2, col1] = v # Symmetric
                except Exception as e:
                    print(f"Warning: Could not calculate for {col1} x {col2}: {str(e)}")
                    corr_matrix.loc[col1, col2] = 0
                    corr_matrix.loc[col2, col1] = 0

    print("Correlation matrix calculated successfully!")
    print("="*70, "\n")
    
    return corr_matrix

In [ ]:
def plot_categorical_correlation_heatmap(corr_matrix, figsize=(18, 16), annot=False, save=False, filename="categorical_correlation.png"):
    """
    Plot a heatmap of categorical correlations using Cramér's V.
    
    Parameters
    ----------
    corr_matrix : pd.DataFrame
        Correlation matrix from categorical_correlation_matrix()
    figsize : tuple, optional
        Figure size (width, height), by default (18, 16)
    annot : bool, optional
        Whether to annotate cells with values, by default False
        Set to True for small matrices (<15 variables)
    save : bool, optional
        Whether to save the plot, by default False
    filename : str, optional
        Filename if saving, by default "categorical_correlation.png"
    
    Examples
    --------
        plot_categorical_correlation_heatmap(cat_corr, figsize=(20, 18), annot=False)
        plot_categorical_correlation_heatmap(cat_corr, annot=True, save=True, filename="cat_corr.png")
    
    Notes
    -----
        - Colour map: YlOrRd (Yellow to Orange to Red)
        - Yellow (0.0): No association
        - Red (1.0): Perfect association
    """
    plt.figure(figsize=figsize)

    # Create heatmap
    sns.heatmap(corr_matrix,
                annot=annot,
                fmt=".2f",
                cmap="RdBu",
                square=True,
                linewidth=0.5,
                cbar_kws={"label": "Cramér's V (0=No Association, 1=Perfect Association)"},
                vmin=0,
                vmax=1
               )
    
    plt.title("Categorical Variables Correlation Matrix (Cramer's V)",
              fontsize=16, fontweight="bold", pad=20)
    plt.xlabel("Features", fontsize=12, fontweight="bold")
    plt.ylabel("Features", fontsize=12, fontweight="bold")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()

    if save:
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        print(f"Plot saved: {filename}")

    plt.show()

    # Print strongest associations
    print("\n" + "="*70)
    print("STRONGEST CATEGORICAL ASSOCIATIONS (Cramér's V)")
    print("="*70)

    # Get upper triangle only (avoid duplicates)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    strong_corr = corr_matrix.where(mask).stack().sort_values(ascending=False)
    
    print("\nTop 20 strongest correlations:")
    print("-"*70)
    for idx, (pair, value) in enumerate(strong_corr.head(20).items(), 1):
        if value > 0.8:
            strength = "Very Strong"
        elif value > 0.6:
            strength = "Strong"
        elif value > 0.4:
            strength = "Moderate"
        elif value > 0.2:
            strength = "Weak"
        else:
            strength = "Very Weak"
        
        print(f"{idx:2d}. {pair[0]:20s} <-> {pair[1]:20s} : {value:.3f} ({strength})")
    print("="*70)

# Calculate categorical correlation matrix
cat_corr = categorical_correlation_matrix(df, cat_cols)

# Display the matrix
print("\nCategorical Correlation Matrix:")
display(cat_corr)

# Plot heatmap (without annotations for better visualization)
plot_categorical_correlation_heatmap(cat_corr, figsize=(18, 16), annot=False)

# Optional: Plot with annotations (if you have few variables)
# plot_categorical_correlation_heatmap(cat_corr, figsize=(20, 18), annot=True)

# Optional: Save the plot
# plot_categorical_correlation_heatmap(cat_corr, save=True, filename="house_prices_cat_correlation.png")

In [ ]:
# Separate target if it exists, otherwise use the whole dataframe for splitting
target_col = "SalePrice"
if target_col in df.columns:
    X = df.drop(target_col, axis=1)
    y = df[target_col]
else:
    X = df.copy()
    y = None

# Split the data: 80% Train, 20% Validation
# Using a fixed random_state ensures reproducibility
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Recombine X and y for the training set only (as our handler learns from the dataframe context)
train_df = X_train.copy()
if y_train is not None:
    train_df[target_col] = y_train

# Validation set (features only)
valid_df = X_valid.copy()
# Note: We do not add the target to valid_df to prevent accidental leakage usage, 
# though for validation evaluation you will use y_valid separately.

print(f"Data Split Complete:")
print(f"Train Shape: {train_df.shape}")
print(f"Valid Shape: {valid_df.shape}")

In [ ]:
def analyse_missing_values(dataframe, plot=True):
    """
    Analyses missing values in the dataframe and displays detailed statistics.
    
    Parameters
    ----------
    dataframe : pandas.DataFrame
        The dataframe to analyse.
    plot : bool, optional
        Whether to visualize the missing data matrix.
        
    Returns
    -------
    missing_df : pandas.DataFrame
        Summary dataframe of missing values sorted by percentage.
    """
    missing_counts = dataframe.isnull().sum()
    missing_pct = 100 * missing_counts / len(dataframe)
    
    missing_df = pd.DataFrame({
        "Missing_Count": missing_counts,
        "Missing_Percentage": missing_pct
    })
    
    missing_df = missing_df[missing_df["Missing_Count"] > 0].sort_values(
        by="Missing_Percentage", ascending=False
    )
    
    print("-" * 50)
    print("MISSING VALUES ANALYSIS")
    print("-" * 50)
    print(f"Total rows: {len(dataframe)}")
    print(f"Columns with missing values: {len(missing_df)} / {len(dataframe.columns)}")
    
    if not missing_df.empty:
        display(missing_df)
        if plot:
            plt.figure(figsize=(12, 6))
            msno.matrix(dataframe)
            plt.title("Missing Values Matrix", fontsize=14)
            plt.show()
    else:
        print("No missing values found.")
        
    return missing_df

In [ ]:
def create_missing_value_handler(train_df, verbose=True):
    """
    Creates a closure (handler function) that encapsulates parameters learnt 
    strictly from the training data. This ensures no data leakage occurs 
    when applying the strategy to validation or test sets.

    This function implements domain-specific strategies for the House Prices dataset:
    - Fills categorical NaNs with "None" (meaning "feature doesn't exist")
    - Fills related numerical NaNs with 0
    - Uses neighborhood-based median for LotFrontage
    - Uses mode for single missing values

    Parameters
    ----------
    train_df : pandas.DataFrame
        The training set used to calculate medians, modes, and other statistics.
    verbose : bool, optional
        Controls the verbosity of the creation process.

    Returns
    -------
    function
        A handler function `apply_handler(dataframe, verbose=True)` that 
        transforms any given dataframe using the pre-learnt parameters.
    """
    
    # Learn parameters from training data
    params = {}
    
    # 1. LotFrontage: Learn neighbourhood-specific medians
    if "LotFrontage" in train_df.columns and "Neighborhood" in train_df.columns:
        params["lot_frontage_by_neighborhood"] = train_df.groupby("Neighborhood")["LotFrontage"].median().to_dict()
        params["lot_frontage_global_median"] = train_df["LotFrontage"].median()

    # 2. Electrical: Learn the mode
    if "Electrical" in train_df.columns:
        # Use series mode; handle case where series might be empty or all NaNs
        modes = train_df["Electrical"].mode()
        params["electrical_mode"] = modes[0] if not modes.empty else None

    if verbose:
        print(f"Handler Created. Learnt parameters from {len(train_df)} training rows.")

    def apply_handler(dataframe, verbose=True):
        """
        Applies the learnt missing value strategies to a target dataframe.
        """
        df = dataframe.copy()
                
        # 1. Categorical: Fill with "None" (Static Rule)
        cat_none_cols = [
            "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
            "GarageType", "GarageFinish", "GarageQual", "GarageCond",
            "Alley", "PoolQC", "MiscFeature", "FireplaceQu", "MasVnrType", "Fence"
        ]
        
        for col in cat_none_cols:
            if col in df.columns:
                df[col] = df[col].fillna("None")

        # 2. Numerical: Fill with 0 (Static Rule)
        num_zero_cols = [
            "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", 
            "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath", 
            "GarageCars", "GarageArea"
        ]
        
        for col in num_zero_cols:
            if col in df.columns:
                df[col] = df[col].fillna(0)

        # 3. LotFrontage: Apply learnt medians
        if "LotFrontage" in df.columns:
            # Vectorised approach is faster, but apply is safer for dictionary lookups with fallbacks
            # Using the closure 'params' variable
            df["LotFrontage"] = df.apply(
                lambda row: params["lot_frontage_by_neighborhood"].get(
                    row["Neighborhood"], 
                    params["lot_frontage_global_median"]
                ) if pd.isna(row["LotFrontage"]) else row["LotFrontage"],
                axis=1
            )

        # 4. GarageYrBlt: Sentinel value (Static Rule)
        if "GarageYrBlt" in df.columns:
            df["GarageYrBlt"] = df["GarageYrBlt"].fillna(-1)

        # 5. Electrical: Apply learnt mode
        if "Electrical" in df.columns and params.get("electrical_mode"):
            df["Electrical"] = df["Electrical"].fillna(params["electrical_mode"])

        if verbose:
            remaining = df.isnull().sum().sum()
            print(f"Transformation complete. Remaining missing values: {remaining}")

        return df

    return apply_handler

# 1. Analyse missing values in the Training Set (Optional but recommended)
print("Analysing Train Set:")
analyse_missing_values(train_df, plot=False)

# 2. Fit: Learn parameters strictly from the Training Set
# This ensures medians/modes from the validation set do not leak into our model.
missing_handler = create_missing_value_handler(train_df, verbose=True)

# 3. Transform: Apply the learned strategy to BOTH sets
# Note: We use the SAME handler object for both.
print("\nProcessing Train Set:")
train_df_clean = missing_handler(train_df, verbose=True)

print("\nProcessing Validation Set:")
valid_df_clean = missing_handler(valid_df, verbose=True)

# 4. Verification
print(f"\nVerification - Missing values in Train Set: {train_df_clean.isnull().sum().sum()}")
print(f"\nVerification - Missing values in Valid Set: {valid_df_clean.isnull().sum().sum()}, \n")

train_cat_cols_clean, train_num_cols_clean, train_cat_but_car_clean, train_num_but_cat_clean = grab_col_names(train_df_clean)
train_num_cols_clean = [col for col in train_num_cols_clean if col not in ["Id"]]

In [ ]:
def calculate_vif(dataframe, num_cols):
    """
    Calculate VIF (Variance Inflation Factor) for numeric features.
    
    VIF measures multicollinearity - how much a feature can be predicted 
    by other features using linear regression.
    
    VIF Interpretation:
    - VIF = 1: No correlation with other features
    - VIF < 5: Low multicollinearity
    - VIF 5-10: Moderate multicollinearity
    - VIF > 10: High multicollinearity
    - VIF = inf: Perfect multicollinearity

    Parameters
    ----------
    dataframe : pd.DataFrame
        The dataframe containing numeric variables (must have NO missing values)
    num_cols : list
        List of numeric column names
    
    Returns
    -------
    pd.DataFrame
        VIF scores sorted by VIF value (descending)
    
    Examples
    --------
        num_cols_for_vif = [col for col in num_cols_clean if col != "SalePrice"]
        vif_results = calculate_vif(df_clean, num_cols_clean)
        display(vif_results)
    
    Notes
    -----
        - Dataframe MUST NOT have missing values (use df_clean, not df)
        - Only works with numeric features (int, float)
        - Features with VIF > 10 indicate redundancy
        - Features with VIF = inf indicate perfect multicollinearity
        - Consider dropping one feature from highly correlated pairs
        
    Warnings
    --------
        If you get "ValueError: array must not contain infs or NaNs":
        - Use df_clean (after missing value handling)
        - NOT df (raw data with missing values)
    """
    # Validate: Check for missing values
    missing_count = dataframe[num_cols].isnull().sum().sum()
    if missing_count > 0:
        raise ValueError(
            f"VIF calculation requires NO missing values! "
            f"Found {missing_count} missing values. "
            f"Use df_clean (after missing value handling) instead of raw df."
        )
    
    # Select only numeric columns
    df_vif = dataframe[num_cols].copy()

    print("="*70)
    print("CALCULATING VIF (VARIANCE INFLATION FACTOR)")
    print("="*70)
    print(f"Number of numeric variables: {len(num_cols)}")
    print(f"Number of observations: {len(df_vif)}")
    print(f"Missing values: {missing_count}\n")

    # Calculate VIF for each numeric feature
    vif_data = pd.DataFrame()
    vif_data["Feature"] = num_cols

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning)
        vif_data["VIF"] = [
            variance_inflation_factor(df_vif.values, i) 
            for i in range(len(num_cols))
        ]
    
    # Sort by VIF (descending)
    vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)
    
    # Add interpretation
    def interpret_vif(vif):
        if np.isinf(vif):
            return "Perfect Multicollinearity"
        elif vif > 10:
            return "High Multicollinearity"
        elif vif > 5:
            return "Moderate Multicollinearity"
        elif vif > 2:
            return "Low Multicollinearity"
        else:
            return "Very Low Multicollinearity"

    vif_data["Interpretation"] = vif_data["VIF"].apply(interpret_vif)
    
    print("VIF calculation completed!")
    print("="*70, "\n")
    
    # Print summary
    perfect = np.isinf(vif_data["VIF"]).sum()
    critical = ((vif_data["VIF"] > 10) & ~np.isinf(vif_data["VIF"])).sum()
    high = ((vif_data["VIF"] > 5) & (vif_data["VIF"] <= 10)).sum()
    moderate = ((vif_data["VIF"] > 2) & (vif_data["VIF"] <= 5)).sum()
    low = (vif_data["VIF"] <= 2).sum()
    
    print("VIF SUMMARY:")
    print("-"*70)
    print(f"Features with VIF = inf (Perfect Multicollinearity):      {perfect:3d}")
    print(f"Features with VIF > 10 (High Multicollinearity):          {critical:3d}")
    print(f"Features with VIF 5-10 (Moderate Multicollinearity):      {high:3d}")
    print(f"Features with VIF 2-5 (Low Multicollinearity):            {moderate:3d}")
    print(f"Features with VIF < 2 (Very Low Multicollinearity):       {low:3d}")
    print("="*70, "\n")
    
    return vif_data

# Exclude target variable (SalePrice) from VIF calculation
num_cols_for_vif = [col for col in train_num_cols_clean if col != "SalePrice"]

# Run VIF analysis on cleaned data
vif_results = calculate_vif(train_df_clean, num_cols_for_vif)

# Display full results
print("\nFULL VIF RESULTS:")
display(vif_results)

# Show PERFECT multicollinearity features (VIF = inf)
print("\nPerfect Multicollinearity (VIF = inf):")
print("-"*70)
perfect_vif = vif_results[np.isinf(vif_results["VIF"])]
display(perfect_vif)

# Show only problematic features (VIF > 10)
print("\nHigh Multicollinearity (VIF > 10):")
print("-"*70)
high_vif = vif_results[(vif_results["VIF"] > 10) & ~np.isinf(vif_results["VIF"])]
display(high_vif)

# Show correlation with target for BOTH perfect and high VIF features
problematic_features = pd.concat([perfect_vif, high_vif])

if len(problematic_features) > 0:    
    print("\nCorrelation with SalePrice (Target):")
    print("-"*70)
    
    for feature in problematic_features["Feature"]:
        corr = train_df_clean[[feature, "SalePrice"]].corr().iloc[0, 1]
        vif_value = vif_results[vif_results["Feature"] == feature]["VIF"].values[0]
        
        if np.isinf(vif_value):
            vif_str = "inf"
        else:
            vif_str = f"{vif_value:>8.2f}"
        
        print(f"{feature:30s} | VIF: {vif_str} | Corr: {corr:>6.3f}")
    print("="*70)
else:
    print("No features with VIF > 10 or VIF = inf.")

In [ ]:
def outlier_thresholds(dataframe, col_name, q1=0.25, q3=0.75):
    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit

def grab_outliers(dataframe, col_name, index=False):
    print("="*50)
    low, up = outlier_thresholds(dataframe, col_name)
    
    # Take a copy of the dataframe and remove NaN values from the columns in the copy.
    # To avoid Pandas errors when detecting outliers.
    # Note: If using df_clean (post-missing value handling), this typically removes 0 rows
    df_no_na = dataframe[dataframe[col_name].notna()]
    
    # Outlier mask
    outlier_mask = (df_no_na[col_name] < low) | (df_no_na[col_name] > up)
    
    if outlier_mask.any():
        print(f"Column: {col_name}")
        print(f"Count of outliers: {outlier_mask.sum()}")
        if outlier_mask.sum() > 10:
            #display(df_no_na[outlier_mask].head())
            pass
        else:
            #display(df_no_na[outlier_mask])
            pass
    else:
        print(f"Column {col_name} has no outlier value")
        pass
        
    if index:
        return df_no_na[outlier_mask].index

cp_num_cols_clean = [col for col in train_num_cols_clean if col != "SalePrice"]

for col in cp_num_cols_clean:
    grab_outliers(train_df_clean, col)

In [ ]:
def hybrid_outlier_detection(dataframe, col_name, contamination=0.05, visualize=True):
    df_numeric = dataframe[col_name].copy()
    # Remove rows with any NaN values (defensive check)
    # Note: If using df_clean (post-missing value handling), this typically removes 0 rows
    df_no_missing = df_numeric.dropna()

    if len(df_no_missing) == 0:
        print("Error: No data left after removing NaN values!")
        return None, None

    print(f"Data shape after removing NaN: {df_no_missing.shape}")
    print(f"Removed {len(df_numeric) - len(df_no_missing)} rows with NaN values\n")

    # 1. IQR Method
    iqr_outliers = set()
    for col in col_name:
        if col in dataframe.columns:
            low, up = outlier_thresholds(dataframe, col)
            mask = ((df_no_missing[col] < low) | (df_no_missing[col] > up)) & df_no_missing[col].notna()
            iqr_outliers.update(df_no_missing[mask].index.tolist())

    # 2 & 3. Isolation Forest + Local Outlier Factor
    scaler = MaxAbsScaler()
    df_scaled = scaler.fit_transform(df_no_missing)
    df_scaled = pd.DataFrame(
        df_scaled,
        columns=df_no_missing.columns,
        index=df_no_missing.index
    )

    # Isolation Forest
    iso_forest = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=100
    )
    iso_pred = iso_forest.fit_predict(df_scaled)
    iso_outliers = set(df_no_missing[iso_pred == -1].index.tolist())

    # Local Outlier Factor
    lof = LocalOutlierFactor(
        n_neighbors=min(20, len(df_no_missing)-1),
        contamination=contamination
    )
    lof_pred =lof.fit_predict(df_scaled)
    lof_scores = lof.negative_outlier_factor_
    lof_outliers = set(df_no_missing[lof_pred == -1].index.tolist())

    ml_consensus_outliers = (iso_outliers | lof_outliers) & iqr_outliers

    outliers_results = pd.DataFrame({
        "IQR": [idx in iqr_outliers for idx in df_no_missing.index],
        "IsolationForest": [idx in iso_outliers for idx in df_no_missing.index],
        "LOF": [idx in lof_outliers for idx in df_no_missing.index],
        "ML_Consensus": [idx in ml_consensus_outliers for idx in df_no_missing.index]
    }, index=df_no_missing.index)

    outliers_results["outlier_count"] = outliers_results[["IQR", "IsolationForest", "LOF"]].sum(axis=1)

    if visualize:
        # 1. LOF Scores
        plt.figure(figsize=(12, 6))
        sorted_scores = np.sort(lof_scores)
        plt.plot(sorted_scores, ".-", markersize=3, linewidth=0.5)
        plt.title("LOF Scores (Sorted)", fontsize=14, fontweight="bold")
        plt.xlabel("Index", fontsize=12)
        plt.ylabel("Negative Outlier Factor", fontsize=12)
        threshold_idx = int(len(sorted_scores) * contamination)
        if threshold_idx < len(sorted_scores):
            plt.axhline(y=sorted_scores[threshold_idx], 
                       color="r", linestyle="--", linewidth=2,
                       label=f"Threshold ({contamination*100}%)")
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # 2. Algorithm Benchmark
        plt.figure(figsize=(10, 6))
        algo_counts = outliers_results[["IQR", "IsolationForest", "LOF"]].sum()
        bars = plt.bar(algo_counts.index, algo_counts.values, 
                      color=["#f92702", "#020af9", "#02f92b"],
                      edgecolor="black", linewidth=1.5)
        plt.title("Outliers Detected by Each Algorithm", fontsize=14, fontweight="bold")
        plt.ylabel("Number of Outliers", fontsize=12)
        plt.xticks(rotation=45, ha="right", fontsize=11)
        
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f"{int(height)}",
                    ha="center", va="bottom", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()
        
        # 3. Consensus Distribution
        plt.figure(figsize=(10,6))
        consensus_counts = outliers_results["outlier_count"].value_counts().sort_index()
        bars = plt.bar(consensus_counts.index, consensus_counts.values,
                      color=["#f92702", "#020af9", "#02f92b", "#640575"],
                      edgecolor="black", linewidth=1.5)
        plt.title("Consensus Distribution", fontsize=14, fontweight="bold")
        plt.xlabel("Number of Algorithms Agreeing", fontsize=12)
        plt.ylabel("Number of Data Points", fontsize=12)
        plt.xticks(range(4), fontsize=11)

        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f"{int(height)}",
                    ha="center", va="bottom", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

        # 4. Venn Diagram
        try:
            from matplotlib_venn import venn3
            plt.figure(figsize=(10, 8))
            venn = venn3([iqr_outliers, iso_outliers, lof_outliers], 
                        ("IQR", "IsoForest", "LOF"),
                        set_colors=("#f92702", "#020af9", "#e1f902"),
                        alpha=0.6)
            
            for text in venn.set_labels:
                if text:
                    text.set_fontsize(14)
                    text.set_fontweight("bold")
            for text in venn.subset_labels:
                if text:
                    text.set_fontsize(12)
                    text.set_fontweight("bold")

            plt.title("Outlier Detection Overlap", fontsize=14, fontweight="bold")
            plt.tight_layout()
            plt.show()
            
        except ImportError:
            print("\nmatplotlib-venn is not installed. To install it: pip install matplotlib-venn")
            # Alternative: Text-based display
            plt.figure(figsize=(10, 8))
            plt.axis("off")
            overlap_text = f"""
            Overlap Statistics:
            
            IQR only: {len(iqr_outliers - iso_outliers - lof_outliers)}
            IsoForest only: {len(iso_outliers - iqr_outliers - lof_outliers)}
            LOF only: {len(lof_outliers - iqr_outliers - iso_outliers)}
            
            IQR ∩ IsoForest: {len(iqr_outliers & iso_outliers)}
            IQR ∩ LOF: {len(iqr_outliers & lof_outliers)}
            IsoForest ∩ LOF: {len(iso_outliers & lof_outliers)}
            
            All three: {len(iqr_outliers & iso_outliers & lof_outliers)}
            
            ML Consensus ((ISO ∪ LOF) ∩ IQR): {len(ml_consensus_outliers)}
            """
            plt.text(0.5, 0.5, overlap_text, fontsize=14, 
                    verticalalignment="center", horizontalalignment="center",
                    family="monospace", bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
            plt.title("Outlier Detection Overlap", fontsize=14, fontweight="bold")
            plt.tight_layout()
            plt.show()

    # Summary statistics
    print("\n" + "="*70)
    print(f"{'Outlier Detection Summary':^70}")
    print("="*70)
    print(f"Total data points: {len(df_no_missing)}")
    print("-"*70)
    print(f"IQR method:{len(iqr_outliers):>4} outliers ({len(iqr_outliers)/len(df_no_missing)*100:>5.2f}%)")
    print(f"Isolation Forest:{len(iso_outliers):>4} outliers ({len(iso_outliers)/len(df_no_missing)*100:>5.2f}%)")
    print(f"LOF:{len(lof_outliers):>4} outliers ({len(lof_outliers)/len(df_no_missing)*100:>5.2f}%)")
    print("-"*70)
    print(f"ML Consensus ((ISO ∪ LOF) ∩ IQR): {len(ml_consensus_outliers):>4} outliers ({len(ml_consensus_outliers)/len(df_no_missing)*100:>5.2f}%)")
    print(f"All 3 agree:{len(iqr_outliers & iso_outliers & lof_outliers):>4} outliers ({len(iqr_outliers & iso_outliers & lof_outliers)/len(df_no_missing)*100:>5.2f}%)")
    print("="*70 + "\n")
    
    return outliers_results, lof_scores, ml_consensus_outliers

In [ ]:
def replace_outliers_with_thresholds(dataframe, col_name, outlier_indices):
    df = dataframe.copy()
    replacement_log = []

    for col in col_name:
        if col not in dataframe.columns:
            continue

        low, up = outlier_thresholds(dataframe, col)
        replaced_count = 0

        for idx in outlier_indices:
            if idx in df.index and pd.notna(df.loc[idx, col]):
                original_value = df.loc[idx, col]
                
                if df.loc[idx, col] < low:
                    df.loc[idx, col] = low
                    replaced_count += 1
                    replacement_log.append({
                        "index": idx,
                        "column": col,
                        "original": original_value,
                        "new": low,
                        "type": "lower"
                    })

                elif df.loc[idx, col] > up:
                    df.loc[idx, col] = up
                    replaced_count += 1
                    replacement_log.append({
                        "index": idx,
                        "column": col,
                        "original": original_value,
                        "new": up,
                        "type": "upper"
                    })

        if replaced_count > 0:
            print(f"  {col}: {replaced_count} value was changed")
    
    return df, pd.DataFrame(replacement_log)

In [ ]:
zeroheavy_summary = []
for col in cp_num_cols_clean:
    zero_count = (train_df_clean[col] == 0).sum()
    zero_pct = zero_count / len(train_df_clean) * 100
    zeroheavy_summary.append({
        'column': col,
        'unique_values': train_df_clean[col].nunique(),
        'zero_count': zero_count,
        'zero_pct': f"{zero_pct:.1f}%",
        'most_common_value': train_df_clean[col].mode()[0] if len(train_df_clean[col].mode()) > 0 else None,
        'most_common_count': train_df_clean[col].value_counts().iloc[0] if len(train_df_clean[col]) > 0 else 0
    })

pd.DataFrame(zeroheavy_summary).sort_values('zero_pct', ascending=False)

In [ ]:
# Exclude zero-inflated variables from outlier detection. For example:
# LowQualFinSF: 98.2% zeros - IQR method doesn't work (Q1=Q3=0, thresholds=0)
# MiscVal: 96.7% zeros - IQR method doesn't work (Q1=Q3=0, thresholds=0)
# 3SsnPorch: 98.35% zeros - IQR method doesn't work (Q1=Q3=0, thresholds=0)
# Goes on for other similar features...
# Also Excluded:
# GarageYrBlt: Many missing values filled with -1 sentinel, not suitable for IQR
# We'll create a binary feature for this in feature engineering phase
zero_inflated_vars = ['3SsnPorch', 'LowQualFinSF', 'MiscVal', 'ScreenPorch', 
                   'BsmtFinSF2', 'EnclosedPorch', 'MasVnrArea', '2ndFlrSF', 
                   'WoodDeckSF', "OpenPorchSF", "GarageYrBlt"]

# Exclude features with natural wide range and meaningful extreme values
# LotArea: Lot size in square feet - naturally has wide distribution
#          IQR upper limit (~17,673) is too restrictive and caps many legitimate large lots
#          99th percentile (~37,568) still insufficient for capturing true variance
#          Extreme values are informative for price prediction (larger lot = potentially higher price)
#          Better handled through feature transformation (log/sqrt) in feature engineering phase
natural_wide_range_vars = ['LotArea']

# Combine exclusion lists
outlier_exclusion_vars = zero_inflated_vars + natural_wide_range_vars

# Remove zero-inflated variables from outlier detection
cp_num_cols_for_outlier = [col for col in cp_num_cols_clean if col not in outlier_exclusion_vars]

print("Starting outlier detection...")
print(f"Features to analyze: {len(cp_num_cols_clean)}")
print(f"Features excluded from outlier detection: {len(outlier_exclusion_vars)}")
print(f"  - Zero-inflated: {len(zero_inflated_vars)}")
print(f"  - Natural wide range: {len(natural_wide_range_vars)}")
print(f"Features for outlier detection: {len(cp_num_cols_for_outlier)}")

outliers_results, lof_scores, ml_consensus_outliers = hybrid_outlier_detection(
    train_df_clean, 
    cp_num_cols_for_outlier, # LowQualFinSF excluded
    contamination=0.02 # By looking at the LOF Score graph, the breakpoint of the elbow was selected.
)

In [ ]:
if outliers_results is not None and len(ml_consensus_outliers) > 0:
    print(f"\n{'='*70}")
    print(f"Replacing {len(ml_consensus_outliers)} outlier rows with IQR thresholds...")
    print(f"{'='*70}\n")
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning)
        train_df_outlier_handling, replacement_log = replace_outliers_with_thresholds(
            train_df_clean, 
            cp_num_cols_for_outlier, 
            ml_consensus_outliers
        )
    
    print(f"\n{'='*70}")
    print(f"✓ Outlier replacement completed!")
    print(f"  Total replacements: {len(replacement_log)}")
    print(f"{'='*70}\n")
    
    # Show the first few replacements
    if len(replacement_log) > 0:
        print("Sample replacements:")
        print(replacement_log.head(10))
else:
    print("\nNo outliers detected or detection failed.")

In [ ]:
cat_cols_outlierhandling , num_cols_outlierhandling, cat_but_car_outlierhandling, num_but_cat_outlierhandling = grab_col_names(train_df_outlier_handling)
num_cols_outlierhandling = [col for col in num_cols_outlierhandling if col not in ["Id"]]

In [ ]:
# ========================================
# FEATURE ENGINEERING
# ========================================

def create_new_features(dataframe):
    """
    Creates new engineered features from existing features.
    
    Parameters:
    -----------
    dataframe : pd.DataFrame
        Input dataframe with original features
        
    Returns:
    --------
    pd.DataFrame
        Dataframe with additional engineered features
    """

    # Create a copy to avoid modifying original dataframe
    df_fe = dataframe.copy()

    # =======================================
    # 1. TOTAL AREAS
    # =======================================
    print("Creating Total Area Features")

    # Total square footage (above ground + basement)
    df_fe["TotalSF"] = df_fe["GrLivArea"] + df_fe["TotalBsmtSF"]

    # Total bathrooms (fullbaths count as 1, halfbaths as 0.5)
    df_fe["TotalBath"] = (df_fe["FullBath"] +
                          0.5 * df_fe["HalfBath"] +
                          df_fe["BsmtFullBath"] +
                          0.5 * df_fe["BsmtHalfBath"])
    
    # Total porch square footage (only porches)
    df_fe["TotalPorchSF"] = (df_fe["OpenPorchSF"] +
                            df_fe["EnclosedPorch"] +
                            df_fe["3SsnPorch"] +
                            df_fe["ScreenPorch"])

    # Total outdoor space (porches + deck)
    df_fe["TotalOutdoorSF"] = df_fe["TotalPorchSF"] + df_fe["WoodDeckSF"]
    
    # ========================================
    # 2. PROPORTIONAL FEATURES (RATIOS)
    # ========================================
    print("Creating Proportional Features")

    # Living area as proportion of lot area
    df_fe["LivingArea_to_LotArea"] = df_fe["GrLivArea"] / df_fe["LotArea"]

    # Basement proportion of living area (if basement exists)
    df_fe["Basement_to_GrLivArea"] = np.where(
        df_fe["GrLivArea"] > 0,
        df_fe["TotalBsmtSF"] / df_fe["GrLivArea"],
        0
    )

    # Garage area as proportion of lot area
    df_fe["GarageArea_to_LotArea"] = np.where(
        df_fe["LotArea"] > 0,
        df_fe["GarageArea"] / df_fe["LotArea"],
        0
    )

    # Rooms per unit area (rooms per square foot)
    df_fe["Rooms_per_Area"] = np.where(
        df_fe["GrLivArea"] > 0,
        (df_fe["TotRmsAbvGrd"] + df_fe["TotalBsmtSF"]) / df_fe["GrLivArea"],
        0
    )

    # Bedroom proportion of total rooms
    df_fe["Bedroom_to_TotRms"] = np.where(
        df_fe["TotRmsAbvGrd"] > 0,
        df_fe["BedroomAbvGr"] / df_fe["TotRmsAbvGrd"],
        0
    )

    # Kitchen proportion of total rooms
    df_fe["Kitchen_to_TotRms"] = np.where(
        df_fe["TotRmsAbvGrd"] > 0,
        df_fe["KitchenAbvGr"] / df_fe["TotRmsAbvGrd"],
        0
    )

    # ========================================
    # 3. AGE FEATURES
    # ========================================
    print("Creating Age Features")

    # House age at time of sale
    df_fe["HouseAge"] = df_fe["YrSold"] - df_fe["YearBuilt"]

    # Time since last remodeling
    df_fe["RemodAge"] = df_fe["YrSold"] - df_fe["YearRemodAdd"]

    # Garage age (handle -1 sentinel value for missing GarageYrBlt)
    df_fe["GarageAge"] = np.where(
        df_fe["GarageYrBlt"] == -1,
        -1,     # Keep -1 for houses without garage
        df_fe["YrSold"] - df_fe["GarageYrBlt"]
    )

    # ========================================
    # 4. BINARY FEATURES (0/1 indicators)
    # ========================================
    print("Creating Binary Features")

    # Was the house remodeled after construction?
    df_fe["IsRemodeled"] = (df_fe["YearRemodAdd"] != df_fe["YearBuilt"]).astype(int)

    # Was the house sold in the same year it was built?
    df_fe["IsNewHouse"] = (df_fe["YrSold"] == df_fe["YearBuilt"]).astype(int)

    # Was the house sold in the same year it was remodeled?
    df_fe["IsNewlyRemodeled"] = (df_fe["YrSold"] == df_fe["YearRemodAdd"]).astype(int)

    # Binary indicators for zero-inflated features
    df_fe["Is3SsnPorch"] = (df_fe["3SsnPorch"] > 0).astype(int)
    df_fe["IsLowQualFinSF"] = (df_fe["LowQualFinSF"] > 0).astype(int)
    df_fe["IsMiscVal"] = (df_fe["MiscVal"] > 0).astype(int)
    df_fe["IsScreenPorch"] = (df_fe["ScreenPorch"] > 0).astype(int)
    df_fe["IsBsmtFinSF2"] = (df_fe["BsmtFinSF2"] > 0).astype(int)
    df_fe["IsEnclosedPorch"] = (df_fe["EnclosedPorch"] > 0).astype(int)
    df_fe["IsMasVnrArea"] = (df_fe["MasVnrArea"] > 0).astype(int)
    df_fe["Is2ndFlrSF"] = (df_fe["2ndFlrSF"] > 0).astype(int)
    df_fe["IsWoodDeckSF"] = (df_fe["WoodDeckSF"] > 0).astype(int)
    df_fe["IsOpenPorchSF"] = (df_fe["OpenPorchSF"] > 0).astype(int)
    df_fe["IsGarage"] = (df_fe["GarageYrBlt"] != -1).astype(int)    # Has Garage

    # ========================================
    # 5. QUALITY MAPPING (Categorical to Numerical)
    # ========================================
    print("Mapping Quality Variables")
    
    # Quality mapping dictionary
    qual_map = {
        'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 
        'None': 0, 'NA': 0, np.nan: 0
    }
    
    # Apply quality mapping
    df_fe["ExterQual_num"] = df_fe["ExterQual"].map(qual_map).fillna(0)
    df_fe["KitchenQual_num"] = df_fe["KitchenQual"].map(qual_map).fillna(0)
    df_fe["BsmtQual_num"] = df_fe["BsmtQual"].map(qual_map).fillna(0)
    df_fe["HeatingQC_num"] = df_fe["HeatingQC"].map(qual_map).fillna(0)
    df_fe["GarageQual_num"] = df_fe["GarageQual"].map(qual_map).fillna(0)

    # Apply condition mapping
    df_fe["ExterCond_num"] = df_fe["ExterCond"].map(qual_map).fillna(0)
    df_fe["BsmtCond_num"] = df_fe["BsmtCond"].map(qual_map).fillna(0)
    df_fe["GarageCond_num"] = df_fe["GarageCond"].map(qual_map).fillna(0)
    # ========================================
    # 6. NUMERICAL QUALITY SCORES
    # ========================================
    print("Creating Quality Score Features")

    # Total quality score (sum of quality ratings)
    df_fe["TotalQual"] = (df_fe["OverallQual"] +
                               df_fe["ExterQual_num"] +
                               df_fe["KitchenQual_num"] +
                               df_fe["BsmtQual_num"] +
                               df_fe["HeatingQC_num"] +
                               df_fe["GarageQual_num"])
    
    # Total condition score (sum of condition ratings)
    df_fe["TotalCond"] = (df_fe["OverallCond"] +
                               df_fe["ExterCond_num"] +
                               df_fe["BsmtCond_num"] +
                               df_fe["GarageCond_num"])
    
    # Interaction: Overall quality * condition
    df_fe["OverallQual_and_OverallCond"] = df_fe["OverallQual"] * df_fe["OverallCond"]

    # Interaction: Living area * quality
    df_fe["LivingArea_and_OverallQuality"] = df_fe["GrLivArea"] * df_fe["OverallQual"]

    # ========================================
    # 7. CATEGORICAL QUALITY COMBINATIONS
    # ========================================
    print("Creating Categorical Quality Combination Features")

    # Combine exterior quality and condition
    df_fe["Exter_Qual_and_Cond"] = df_fe["ExterQual"].astype(str) + "_" + df_fe["ExterCond"].astype(str)
    
    # Combine basement quality and condition
    df_fe["Bsmt_Qual_and_Cond"] = df_fe["BsmtQual"].astype(str) + "_" + df_fe["BsmtCond"].astype(str)
    
    # Combine heating type and quality
    df_fe["Heating_and_HeatingQC"] = df_fe["Heating"].astype(str) + "_" + df_fe["HeatingQC"].astype(str)
    
    # Combine garage quality and condition
    df_fe["Garage_Qual_and_Cond"] = df_fe["GarageQual"].astype(str) + "_" + df_fe["GarageCond"].astype(str)

    # ========================================
    # 8. QUALITY GROUPING
    # ========================================

    print("Creating Quality Grouping Features")

    # Overall quality grouping
    df_fe["QualityLevel"] = pd.cut(
        df_fe["OverallQual"],
        bins=[0, 2, 4, 5, 7, 10],
        labels=["Very Poor", "Poor", "Average", "Good", "Excellent"],
        include_lowest=True
    )

    # Overall condition grouping
    df_fe["ConditionLevel"] = pd.cut(
        df_fe["OverallCond"],
        bins=[0, 2, 4, 5, 7, 10],
        labels=["Very Poor", "Poor", "Average", "Good", "Excellent"],
        include_lowest=True
    )

    # ========================================
    # 9. LUXURY AND FEATURE SCORES
    # ========================================
    print("Creating Luxury and Feature Score Features")

    # Basic amenities binary features
    df_fe["HasPool"] = (df_fe["PoolArea"] > 0).astype(int)
    df_fe["HasFireplace"] = (df_fe["Fireplaces"] > 0).astype(int)
    df_fe["HasGarage"] = (df_fe["GarageArea"] > 0).astype(int)
    df_fe["HasBasement"] = (df_fe["TotalBsmtSF"] > 0).astype(int)
    df_fe["HasPorch"] = (df_fe["TotalPorchSF"] > 0).astype(int)

    # Basic amenities score
    df_fe["FeatureScore"] = (df_fe["HasPool"] +
                            df_fe["HasFireplace"] +
                            df_fe["HasGarage"] +
                            df_fe["HasBasement"] +
                            df_fe["HasPorch"])
    
    # Luxury score (amenities weighted by luxury level)
    df_fe["LuxuryScore"] = ((df_fe["PoolArea"] > 0).astype(int) * 3 + 
                             (df_fe["Fireplaces"] > 1).astype(int) * 2 + 
                             (df_fe["GarageCars"] > 2).astype(int) * 2 + 
                             (df_fe["OverallQual"] >= 8).astype(int) * 3)
    
    # ========================================
    # 10. SEASON FEATURE
    # ========================================
    print("Creating Season Feature")

    # Season of sale
    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Fall", 10: "Fall", 11: "Fall"
    }

    df_fe["Season"] = df_fe["MoSold"].map(season_map)

    # ========================================
    # 11. SALE TYPE GROUPING
    # ========================================
    print("Creating Sale Type Grouping Features")
    
    # Sale financing type grouping
    sale_financing_map = {
        "WD": "Standard_Deed", "CWD": "Standard_Deed", "VWD": "Standard_Deed",
        "Con": "Contract_Sale", "ConLw": "Contract_Sale", "ConLI": "Contract_Sale", 
        "ConLD": "Contract_Sale",
        "New": "New_Construction",
        "COD": "Distressed_Special", "Oth": "Distressed_Special"
    }
    
    df_fe['Sale_Financing_Type'] = df_fe['SaleType'].map(sale_financing_map)
    
    # Sale type risk grouping
    sale_risk_map = {
        "WD": "Low_Risk", "CWD": "Low_Risk", "VWD": "Low_Risk", "New": "Low_Risk",
        "Con": "Medium_Risk", "ConLI": "Medium_Risk", "ConLD": "Medium_Risk",
        "ConLw": "High_Risk", "COD": "High_Risk", "Oth": "High_Risk"
    }
    
    df_fe['SaleType_Risk'] = df_fe['SaleType'].map(sale_risk_map)
    
    # ========================================
    # 12. SALE CONDITION GROUPING
    # ========================================
    print("Creating Sale Condition Grouping Features")
    
    # Sale market condition grouping
    sale_market_map = {
        "Normal": "Normal_Complete",
        "Abnorml": "Abnormal_Distressed", "Family": "Abnormal_Distressed",
        "Partial": "Partial_Development", "AdjLand": "Partial_Development", 
        "Alloca": "Partial_Development"
    }
    
    df_fe['Sale_Market_Condition'] = df_fe['SaleCondition'].map(sale_market_map)
    
    # Sale condition risk grouping
    sale_cond_risk_map = {
        'Normal': 'Low_Risk',
        'AdjLand': 'Medium_Risk', 'Alloca': 'Medium_Risk', 'Partial': 'Medium_Risk',
        'Abnorml': 'High_Risk', 'Family': 'High_Risk'
    }
    
    df_fe['SaleCondition_Risk'] = df_fe['SaleCondition'].map(sale_cond_risk_map)
    
    # ========================================
    # 13. NUMERICAL TRANSFORMATIONS
    # ========================================
    print("Creating Numerical Transformations")

    # Log transformation for LotArea (handle skewness)
    df_fe['LotArea_Log'] = np.log1p(df_fe['LotArea'])  # log1p = log(1+x) to handle zeros
    
    # Square root transformation for LotArea
    df_fe['LotArea_Sqrt'] = np.sqrt(df_fe['LotArea'])

    # ========================================
    # 14. NEIGHBORHOOD GROUPING
    # ========================================
    print("Creating Neighbourhood Grouping Features")

    # Neighborhood price tier grouping (based on domain knowledge/EDA)
    neighborhood_tiers = {
        "Luxury": ["NoRidge", "NridgHt", "StoneBr"],
        "Upper_Middle": ["Timber", "Veenker", "Somerst", "ClearCr", "Crawfor"],
        "Middle": ["CollgCr", "Blmngtn", "Gilbert", "NWAmes", "SawyerW"],
        "Lower_Middle": ["Mitchel", "NAmes", "NPkVill", "SWISU", "Blueste", 
                        "Sawyer", "OldTown", "Edwards", "BrkSide"],
        "Budget": ["BrDale", "IDOTRR", "MeadowV"]
    }

    # Reverse mapping
    neighborhood_price_map = {neighborhood: tier 
                            for tier, neighborhoods in neighborhood_tiers.items() 
                            for neighborhood in neighborhoods}

    df_fe["Neighborhood_Price_Level"] = df_fe["Neighborhood"].map(neighborhood_price_map).fillna("Other")

    # Validate mapping
    unmapped = df_fe[df_fe["Neighborhood_Price_Level"] == "Other"]["Neighborhood"].unique()
    if len(unmapped) > 0:
        print(f"Warning: {len(unmapped)} neighborhoods not mapped: {list(unmapped)}")
    else:
        print("All neighborhoods successfully mapped")

    print(f"\nFeature Engineering Complete!")
    print(f"Original features: {len(dataframe.columns)}")
    print(f"New features added: {len(df_fe.columns) - len(dataframe.columns)}")
    print(f"Total features: {len(df_fe.columns)}")
    
    return df_fe

In [ ]:
# Apply feature engineering to cleaned train dataframe
print("Starting Feature Engineering on Training Data...\n")
train_df_engineered = create_new_features(train_df_outlier_handling)

# Apply feature engineering to validation dataframe
print("\nStarting Feature Engineering on Validation Data...\n")
valid_df_engineered = create_new_features(valid_df_clean)

# Display summary
print("\n" + "="*50)
print("FEATURE ENGINEERING SUMMARY")
print("="*50)
check_df(train_df_engineered)

# Get new feature names
original_cols = set(train_df_outlier_handling.columns)
new_cols = set(train_df_engineered.columns) - original_cols
print(f"\nNewly Created Features ({len(new_cols)}):")
print("-" * 50)
for i, col in enumerate(sorted(new_cols), 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Train dataframe column names after feature engineering
train_cat_cols_engineered, train_num_cols_engineered, train_cat_but_car_engineered, train_num_but_cat_engineered = grab_col_names(train_df_engineered)
train_num_cols_engineered = [col for col in train_num_cols_engineered if col not in ["Id"]]

# Validation dataframe column names after feature engineering
valid_cat_cols_engineered , valid_num_cols_engineered, valid_cat_but_car_engineered , valid_num_but_cat_engineered  = grab_col_names(valid_df_engineered)
valid_num_cols_engineered = [col for col in valid_num_cols_engineered if col not in ["Id"]]

In [ ]:
def target_encode(X_train, X_val, col, y_train, n_splits=5):
    """
    Applies K-Fold Target Encoding to a single categorical column to prevent data leakage.

    Methodology:
    1. For Training Data (X_train): Uses K-Fold cross-validation logic. The data is split 
       into K folds. For each fold, the mean is calculated using the OTHER (K-1) folds. 
       This prevents the model from "seeing" the target of the row it is currently predicting.
    2. For Validation Data (X_val): Uses the standard global mean from the entire X_train. 
       This mimics the production environment where we use all available history.

    Parameters
    ----------
    X_train : pd.DataFrame
        Training feature set.
    X_val : pd.DataFrame
        Validation feature set.
    col : str
        The name of the column to encode.
    y_train : pd.Series
        The target variable.
    n_splits : int, optional
        Number of folds for K-Fold encoding (default is 5).

    Returns
    -------
    X_train_encoded, X_val_encoded : pd.Series
        The column transformed into numerical mean values.
    """
    # Initialize empty Series for the encoded training column with the same index as X_train
    X_train_encoded = pd.Series(np.nan, index=X_train.index)
    
    # Initialize KFold
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # --- 1. K-Fold Encoding for X_train (Prevents Leakage) ---
    for train_idx, map_idx in kf.split(X_train):
        # Split internal training data into "inner train" and "inner hold-out"
        X_fold_train = X_train.iloc[train_idx]
        y_fold_train = y_train.iloc[train_idx]
        X_fold_map = X_train.iloc[map_idx]
        
        # Calculate means on the "inner train" folds
        # Using groupby on y_train directly aligned with X_train is faster and cleaner
        fold_means = y_fold_train.groupby(X_fold_train[col]).mean()
        
        # Map these means to the "inner hold-out" fold
        X_train_encoded.iloc[map_idx] = X_fold_map[col].map(fold_means)

    # --- 2. Standard Encoding for X_val (Production Logic) ---
    # For validation, we use the entire training set statistics
    global_mean = y_train.mean()
    full_means = y_train.groupby(X_train[col]).mean()
    
    X_val_encoded = X_val[col].map(full_means)
    
    # --- 3. Handle Missing Values (Unseen Categories) ---
    # Fill NaNs in both sets with the global mean of the target
    X_train_encoded.fillna(global_mean, inplace=True)
    X_val_encoded.fillna(global_mean, inplace=True)
    
    return X_train_encoded, X_val_encoded


def prepare_features_for_models(X_train, X_val, y_train, cat_cols, cat_but_car):
    """
    Prepares features for modelling by handling encoding, scaling, and feature tracking.
    
    Methodology:
    1. High Cardinality Columns: Applies K-Fold Target Encoding (Updated).
    2. Low Cardinality Columns: Applies One-Hot Encoding.
    3. Scaling: Standardizes features (Z-score) for linear models (ElasticNet).
    4. Mapping: Creates a dictionary to map dummy variables back to their original parent feature.

    Parameters
    ----------
    X_train, X_val : pd.DataFrame
        Raw training and validation split data.
    y_train : pd.Series
        Target variable for target encoding.
    cat_cols : list
        List of all categorical columns.
    cat_but_car : list
        List of high-cardinality categorical columns (to be target encoded).

    Returns
    -------
    X_train_scaled, X_val_scaled : pd.DataFrame
        Processed and scaled data ready for ElasticNet.
    feature_names : list
        List of final column names after encoding.
    scaler : StandardScaler
        The fitted scaler object.
    feature_map : dict
        A dictionary mapping original feature names to their encoded dummy columns.
        Structure: {'Original_Col': ['Dummy_Col_1', 'Dummy_Col_2', ...]}
    """
    X_train_proc = X_train.copy()
    X_val_proc = X_val.copy()
    
    # Feature Map: {Original_Col_Name: [List_of_Encoded_Col_Names]}
    feature_map = {}
    
    # Identify column types
    cat_but_car_in_data = [col for col in cat_but_car if col in X_train.columns]
    low_card_cat = [col for col in cat_cols if col in X_train.columns and col not in cat_but_car_in_data]
    
    # 1. Target encoding for high cardinality (1-to-1 mapping)
    for col in cat_but_car_in_data:
        # Calls the updated K-Fold enabled function
        X_train_proc[col], X_val_proc[col] = target_encode(X_train, X_val, col, y_train)
        feature_map[col] = [col]
    
    # 2. One-hot encoding for low cardinality (1-to-Many mapping)
    for col in low_card_cat:
        dummies_train = pd.get_dummies(X_train_proc[col], prefix=col, drop_first=True)
        dummies_val = pd.get_dummies(X_val_proc[col], prefix=col, drop_first=True)
        
        # Align columns (ensure val set has same dummies as train)
        for c in dummies_train.columns:
            if c not in dummies_val.columns:
                dummies_val[c] = 0
        dummies_val = dummies_val[dummies_train.columns]
        
        X_train_proc = pd.concat([X_train_proc.drop(col, axis=1), dummies_train], axis=1)
        X_val_proc = pd.concat([X_val_proc.drop(col, axis=1), dummies_val], axis=1)
        
        # Record which dummy columns belong to this original categorical feature
        feature_map[col] = list(dummies_train.columns)
    
    # 3. Numerical columns (1-to-1 mapping)
    # Any column not in cat_cols is considered numerical for this mapping
    num_cols = [c for c in X_train.columns if c not in cat_cols]
    for col in num_cols:
        feature_map[col] = [col]

    # 4. Scale for ElasticNet
    scaler = MaxAbsScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_proc),
        columns=X_train_proc.columns,
        index=X_train_proc.index
    )
    X_val_scaled = pd.DataFrame(
        scaler.transform(X_val_proc),
        columns=X_val_proc.columns,
        index=X_val_proc.index
    )
    
    return X_train_scaled, X_val_scaled, list(X_train_proc.columns), scaler, feature_map


def calculate_grouped_permutation(model, X_val, y_val, feature_map):
    """
    Calculates Permutation Importance and aggregates results for One-Hot Encoded features.
    
    Methodology:
    1. Calculates standard Permutation Importance (on Validation set).
    2. Uses 'feature_map' to identify which dummy columns belong to the same original feature.
    3. Sums the importance scores of these dummy groups to get a single score per original feature.
    
    Parameters
    ----------
    model : fitted estimator
        The trained model (ElasticNet, RF, or XGB).
    X_val : pd.DataFrame
        Validation features.
    y_val : pd.Series
        Validation targets.
    feature_map : dict
        Mapping of original features to encoded columns.

    Returns
    -------
    pd.Series
        Aggregated importance scores sorted descending.
    """
    result = permutation_importance(
        model, X_val, y_val, n_repeats=5, random_state=42, n_jobs=-1, scoring='r2'
    )
    raw_imp = pd.Series(result.importances_mean, index=X_val.columns)
    
    grouped_imp = {}
    for original, encoded_cols in feature_map.items():
        valid_cols = [c for c in encoded_cols if c in X_val.columns]
        if valid_cols:
            grouped_imp[original] = raw_imp[valid_cols].sum()
            
    return pd.Series(grouped_imp).sort_values(ascending=False)


def train_and_get_importances(X_train, X_val, y_train, y_val, feature_map):
    """
    Trains ElasticNet, RandomForest, and XGBoost models and computes their importances.
    
    Methodology:
    Trains each model on the processed training set, evaluates it on the validation set,
    and then calls 'calculate_grouped_permutation' to derive unbiased feature importance.

    Returns
    -------
    results : dict
        A dictionary containing the trained models, their metrics (RMSE, R2), 
        and the calculated feature importance Series.
    """
    results = {}
    
    # 1. ElasticNet
    print("Training ElasticNet...", end=" ")
    elasticnet = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
        cv=5, random_state=42, max_iter=10000
    )
    elasticnet.fit(X_train, y_train)
    y_pred_en = elasticnet.predict(X_val)
    
    print("Calculating Importance...", end=" ")
    imp_en = calculate_grouped_permutation(elasticnet, X_val, y_val, feature_map)

    results['ElasticNet'] = {
        'model': elasticnet,
        'importances': imp_en,
        'metrics': {
            'RMSE': np.sqrt(mean_squared_error(y_val, y_pred_en)),
            'MAE': mean_absolute_error(y_val, y_pred_en),
            'R2': r2_score(y_val, y_pred_en)
        }
    }
    print(f"R²: {results['ElasticNet']['metrics']['R2']:.4f}")
    
    # 2. Random Forest
    print("Training RandomForest...", end=" ")
    rf = RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_split=5, 
        min_samples_leaf=2, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    
    print("Calculating Importance...", end=" ")
    imp_rf = calculate_grouped_permutation(rf, X_val, y_val, feature_map)

    results['RandomForest'] = {
        'model': rf,
        'importances': imp_rf,
        'metrics': {
            'RMSE': np.sqrt(mean_squared_error(y_val, y_pred_rf)),
            'MAE': mean_absolute_error(y_val, y_pred_rf),
            'R2': r2_score(y_val, y_pred_rf)
        }
    }
    print(f"R²: {results['RandomForest']['metrics']['R2']:.4f}")
    
    # 3. XGBoost
    print("Training XGBoost...", end=" ")
    xgb_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05, 
        min_child_weight=3, random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_val)
    
    print("Calculating Importance...", end=" ")
    imp_xgb = calculate_grouped_permutation(xgb_model, X_val, y_val, feature_map)

    results['XGBoost'] = {
        'model': xgb_model,
        'importances': imp_xgb,
        'metrics': {
            'RMSE': np.sqrt(mean_squared_error(y_val, y_pred_xgb)),
            'MAE': mean_absolute_error(y_val, y_pred_xgb),
            'R2': r2_score(y_val, y_pred_xgb)
        }
    }
    print(f"R²: {results['XGBoost']['metrics']['R2']:.4f}")
    
    return results


def plot_model_performance(results):
    """
    Visualizes the comparative performance (RMSE, MAE, R2) of the three models using bar charts.
    """
    models = list(results.keys())
    metrics_names = ['RMSE', 'MAE', 'R2']
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    colours = ['#6d0972', '#720943', '#380972']
    
    for idx, metric in enumerate(metrics_names):
        values = [results[m]['metrics'][metric] for m in models]
        bars = axes[idx].bar(models, values, color=colours, edgecolor='black', linewidth=1.2)
        axes[idx].set_title(metric, fontsize=14, fontweight='bold')
        axes[idx].set_ylabel(metric, fontsize=11)
        
        for bar, val in zip(bars, values):
            height = bar.get_height()
            axes[idx].text(
                bar.get_x() + bar.get_width() / 2., height,
                f'{val:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold'
            )
        
        axes[idx].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def plot_feature_importance_on_ax(importances, model_name, metrics, ax, top_n=30, colour='#667EEA'):
    """
    Helper function to plot feature importance on a specific Matplotlib Axis (ax).
    Used to create the side-by-side subplot layout.
    """
    sorted_imp = importances.sort_values(ascending=True).tail(top_n)
    
    # Plot horizontal bars
    bars = ax.barh(range(len(sorted_imp)), sorted_imp.values, color=colour, edgecolor='black', linewidth=0.5)
    
    # Formatting
    ax.set_yticks(range(len(sorted_imp)))
    ax.set_yticklabels(sorted_imp.index, fontsize=9)
    ax.set_xlabel('Permutation Importance', fontsize=10)
    
    # Title with metrics
    metrics_str = f"R²: {metrics['R2']:.3f} | RMSE: {metrics['RMSE']:.2f} | MAE: {metrics['MAE']:.2f}"
    ax.set_title(f'{model_name}\n{metrics_str}', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)


def get_styled_importance_table(importances, model_name):
    """
    Prepares a dataframe for HTML display, resetting index and sorting values.
    """
    df = pd.DataFrame({
        'Feature': importances.index,
        'Importance': importances.values
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    
    df.index = df.index + 1
    return df


def compare_feature_importance(
    X_train, X_val, y_train, y_val,
    cat_cols=None, cat_but_car=None,
    top_n=30, show_tables=True
):
    """
    Main Execution Function.
    
    Orchestrates the entire pipeline:
    1. Prepares data (Encoding/Scaling).
    2. Trains 3 models (ElasticNet, RF, XGB).
    3. Calculates Grouped Permutation Importance.
    4. Plots performance metrics.
    5. Plots feature importance graphs side-by-side.
    6. Displays detailed feature importance tables side-by-side.
    
    Parameters
    ----------
    X_train, X_val : pd.DataFrame
        Data splits.
    top_n : int
        Number of top features to display in the graphs.
    show_tables : bool
        Whether to render the detailed HTML tables.
        
    Returns
    -------
    results : dict
        Contains all trained model objects, metrics, and importance scores.
    """
    print("=" * 60)
    print(f"{'Feature Importance Comparison':^60}")
    print("=" * 60)
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    
    # Handle None inputs
    if cat_cols is None:
        cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    if cat_but_car is None:
        cat_but_car = []
    
    # Prepare features
    X_train_proc, X_val_proc, feature_names, scaler, feature_map = prepare_features_for_models(
        X_train, X_val, y_train, cat_cols, cat_but_car
    )
    
    # Train models
    results = train_and_get_importances(X_train_proc, X_val_proc, y_train, y_val, feature_map)
    
    # 1. Plot Performance
    print("\n")
    plot_model_performance(results)
    
    # 2. Plot Feature Importances (Side-by-Side 1x3 Grid)
    print("\n" + "="*60)
    print("Top Feature Importances (Grouped Permutation)")
    print("="*60)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, max(10, top_n * 0.3))) # Dynamic height
    colours = {'ElasticNet': '#6d0972', 'RandomForest': '#720943', 'XGBoost': '#380972'}
    
    for (model_name, data), ax in zip(results.items(), axes):
        plot_feature_importance_on_ax(
            data['importances'],
            model_name,
            data['metrics'],
            ax=ax,
            top_n=top_n,
            colour=colours[model_name]
        )
    
    plt.tight_layout()
    plt.show()

    # 3. Display Tables (Side-by-Side HTML)
    if show_tables:
        print("\n" + "="*60)
        print("Detailed Importance Tables (All Features)")
        print("="*60)
        
        html_str = ""
        for model_name, data in results.items():
            df = get_styled_importance_table(data['importances'], model_name)
            
            # Count Non-zeros
            non_zero = (df['Importance'] > 0).sum()
            total = len(df)
            
            # Create a header for the HTML table
            header_html = f"""
            <div style="display:inline-block; vertical-align:top; margin-right:20px;">
            <h3>{model_name}</h3>
            <p>Total: {total} | Non-Zero: {non_zero}</p>
            """
            
            # Convert DF to HTML with specific float format
            table_html = df.to_html(float_format=lambda x: f'{x:.5f}')
            
            # Combine
            html_str += header_html + table_html + "</div>"
        
        # Display all 3 tables horizontally
        display_html(html_str, raw=True)
    
    return results

In [ ]:
# Prepare the Source Data (Using ONLY the Training Data)
feature_selection_df = train_df_engineered.copy()

# Separate Features and Target
X_fs = feature_selection_df.drop(["SalePrice", "Id"], axis=1, errors='ignore')
y_fs = feature_selection_df["SalePrice"]

# Internal Split for Feature Selection
# We create a temporary validation set strictly for calculating Permutation Importance.
# This keeps your original 'valid_df_engineered' pure and untouched.
X_train_int, X_val_int, y_train_int, y_val_int = train_test_split(
    X_fs, 
    y_fs, 
    test_size=0.20, 
    random_state=42
)

print(f"Internal Selection Split:")
print(f"Internal Train Shape: {X_train_int.shape}")
print(f"Internal Valid Shape: {X_val_int.shape}")

# Run the Comparison Pipeline
# We pass the engineered column lists we grabbed earlier.
# Note: cat_but_car_engineered lists might need updating if columns were dropped, 
# but usually, they remain consistent within the train set.

fs_results = compare_feature_importance(
    X_train_int, 
    X_val_int, 
    y_train_int, 
    y_val_int,
    cat_cols=train_cat_cols_engineered,   # Derived from grab_col_names on train_df_engineered
    cat_but_car=train_cat_but_car_engineered,
    top_n=30,
    show_tables=True
)

In [ ]:
def analyse_feature_consensus(results, top_n_display=50):
    """
    Analyzes feature selection consensus across multiple models using Rank Aggregation.
    It does NOT retrain models; it only processes the 'results' dictionary.
    
    Methodology:
    1. Calculates cumulative importance for each model to visualize feature sufficiency.
    2. Ranks features in each model (1 = Most Important).
    3. Computes a 'Consensus Score' by averaging the ranks across models.
       Formula: Score = (Rank_Model1 + Rank_Model2 + Rank_Model3) / 3
    4. Generates a sorted table of features based on this consensus score (Lower is better).
    
    Parameters
    ----------
    results : dict
        Dictionary containing model results from 'compare_feature_importance'.
        Expected structure:
        {
            'ModelName': {'importances': pd.Series(values, index=feature_names), ...},
            ...
        }
    top_n_display : int, optional
        Number of top features to display in the final consensus table. Default is 50.
        
    Returns
    -------
    pd.DataFrame
        The consensus table containing individual ranks and the final score.
    """
    
    model_names = list(results.keys())
    
    # --- Part 1: Cumulative Importance Plot ---
    plt.figure(figsize=(14, 8))
    
    # Color palette for distinct lines
    colors = ['#1f77b4', '#cce206', '#2ca02c', '#d62728', '#9467bd']
    max_features = 0
    
    for i, model in enumerate(model_names):
        # Extract importances and sort descending
        imp = results[model]['importances'].sort_values(ascending=False)
        
        # Clip negative importances to 0 for cleaner cumulative plot
        imp_plot = imp.clip(lower=0)
        
        # Normalize to 0-1 range
        total_imp = imp_plot.sum()
        if total_imp > 0:
            cumulative_imp = imp_plot.cumsum() / total_imp
        else:
            cumulative_imp = pd.Series(0, index=imp.index)
            
        n_features = len(cumulative_imp)
        max_features = max(max_features, n_features)
        
        # Plot the line
        plt.plot(
            np.arange(1, n_features + 1), 
            cumulative_imp.values, 
            label=f"{model}",
            linewidth=2.5,
            color=colors[i % len(colors)],
            alpha=0.8
        )

    # X-axis: Intervals of 10
    x_ticks = np.arange(0, max_features + 10, 10)
    plt.xticks(x_ticks, rotation=45)
    plt.xlabel("Number of Features Selected", fontsize=12)
    
    # Y-axis: Specific intervals (0.7, 0.8, 0.9, ...)
    y_ticks = [0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]
    plt.yticks(y_ticks)
    plt.ylabel("Cumulative Importance (Normalized)", fontsize=12)
    plt.ylim(0, 1.05)
    
    # Threshold lines for 90% and 95%
    plt.axhline(y=0.90, color='r', linestyle='--', alpha=0.6, label='90% Threshold')
    plt.axhline(y=0.95, color='purple', linestyle='--', alpha=0.6, label='95% Threshold')
    
    plt.title("Cumulative Feature Importance by Model", fontsize=14, fontweight='bold')
    plt.grid(True, which='both', linestyle='--', alpha=0.4)
    plt.legend(loc='lower right', frameon=True)
    plt.tight_layout()
    plt.show()
    
    # --- Part 2: Rank Aggregation Table ---
    
    rank_data = {}
    
    for model in model_names:
        # Sort features by importance (Highest to Lowest)
        imp = results[model]['importances'].sort_values(ascending=False)
        
        # Assign Rank: 1 = Best Feature
        # Use existing index (feature names) to align later
        ranks = pd.Series(np.arange(1, len(imp) + 1), index=imp.index, name=f'Rank_{model}')
        rank_data[model] = ranks

    # Merge all model ranks into a single DataFrame
    # Outer join ensures no feature is lost if models have slightly different feature sets
    df_ranks = pd.DataFrame(rank_data)
    
    # Handle missing features (if any): Fill with worst rank + 1
    max_rank_possible = df_ranks.max().max()
    df_ranks = df_ranks.fillna(max_rank_possible + 1)
    
    # Calculate Consensus Score: Average of Ranks
    # Example: Feature A ranks (6, 24, 27) -> Score = 19
    df_ranks['Score'] = df_ranks.mean(axis=1)
    
    # Sort by Score (Lower score is better)
    df_ranks = df_ranks.sort_values('Score', ascending=True)
    
    # Prepare the display table
    output_table = df_ranks.head(top_n_display).copy()
    output_table = output_table.reset_index().rename(columns={'index': 'Feature'})
    
    # Add Overall Rank column
    output_table.insert(0, 'Rank_Overall', np.arange(1, len(output_table) + 1))
    
    # Round the score for readability
    output_table['Score'] = output_table['Score'].round(2)
    
    # Print the table
    print(f"\n{'='*80}")
    print(f"TOP {top_n_display} FEATURES (RANK AGGREGATION CONSENSUS)")
    print(f"{'='*80}")
    # Display logic checks for IPython environment to use pretty HTML table if available
    try:
        from IPython.display import display
        # Format Score column to 2 decimal places (only for this display)
        display(output_table.style.hide(axis='index').format({'Score': '{:.2f}'}))
    except ImportError:
        print(output_table.to_string(index=False))

    return df_ranks

df_ranks = analyse_feature_consensus(fs_results, top_n_display=70)

In [ ]:
# List of top 30 features obtained from Permutation Importance
selected_features = df_ranks.head(30).index.tolist()

print(f"Selected Features ({len(selected_features)}):")
print(selected_features)

# Create Train Final: Includes Features + Target
train_df_final = train_df_engineered[selected_features + ["SalePrice"]].copy()

# Create Validation Final: Includes Features ONLY (SalePrice is stored in y_val separate variable)
valid_df_final = valid_df_engineered[selected_features].copy()

print(f"\nShape Checks:")
print(f"Train Final: {train_df_final.shape}")
print(f"Valid Final: {valid_df_final.shape}")

check_df(train_df_final)

In [ ]:
train_final_cat_cols, train_final_num_cols, train_final_cat_but_car, train_final_num_but_cat = grab_col_names(train_df_final)
train_final_num_cols = [col for col in train_final_num_cols if col not in "Id"]

valid_final_cat_cols, valid_final_num_cols, valid_final_cat_but_car, valid_final_num_but_cat = grab_col_names(valid_df_final)
valid_final_num_cols = [col for col in valid_final_num_cols if col not in "Id"]

In [ ]:
def encode_and_scale(
    X, 
    y=None,
    fitted_objects=None,
    cardinality_threshold=20,
    n_splits=5,
    scaler_type='maxabs',
    random_state=42
):
    """
    Encodes categorical features and scales numerical features.
    
    Training Mode (fitted_objects=None): Fits encoders/scaler and transforms data.
        I. Binary → Label Encoding (0/1)
        II. Low cardinality (< threshold) → One-Hot Encoding
        III. High cardinality (≥ threshold) → KFold Target Encoding
        IV.  Scaling → Applied ONLY to numerical and target-encoded features (preserves sparsity).
    
    Test Mode (fitted_objects provided): Transforms data using fitted objects.
        - Handles unseen categories in One-Hot encoding via reindexing.
        - Applies learned scaling parameters to appropriate columns.

    Parameters
    ----------
    X : pd.DataFrame
        Input features (without target variable)
    y : pd.Series, optional
        Target variable (required for training mode)
    fitted_objects : dict, optional
        Pre-fitted encoders/scaler (None = training, dict = test)
    cardinality_threshold : int, default=20
        Threshold for one-hot vs target encoding
    n_splits : int, default=5
        KFold splits for target encoding
    scaler_type : str, default='maxabs'
        Type of scaler: 'maxabs', 'standard', 'robust', 'minmax'
    
    Returns
    -------
    X_processed : pd.DataFrame
    fitted_objects : dict (training mode only)
    
    Examples
    --------
    >>> # Training mode
    >>> X_train_proc, fitted_obj = encode_and_scale(X_train, y_train)
    >>> 
    >>> # Test mode (using same encoders/scaler)
    >>> X_test_proc = encode_and_scale(X_test, fitted_objects=fitted_obj)
    """
    X_proc = X.copy()
    is_training_mode = fitted_objects is None
    
    if is_training_mode and y is None:
        raise ValueError("y (target) is required for training mode")
    
    if is_training_mode:
        # Auto-detect categorical and numerical columns
        cat_cols = X_proc.select_dtypes(include=['object', 'category']).columns.tolist()
        num_cols = X_proc.select_dtypes(include=['number']).columns.tolist()
        
        # Separate binary, low, and high cardinality
        binary_cols = []
        low_card_cols = []
        high_card_cols = []
        
        for col in cat_cols:
            n_unique = X_proc[col].nunique()
            if n_unique == 2:
                binary_cols.append(col)
            elif n_unique < cardinality_threshold:
                low_card_cols.append(col)
            else:
                high_card_cols.append(col)
        
        print(f"Categorical columns: {len(cat_cols)}")
        print(f"  - Binary: {len(binary_cols)} (label encode to 0/1)")
        print(f"  - One-Hot: {len(low_card_cols)} (cardinality < {cardinality_threshold})")
        print(f"  - Target (KFold): {len(high_card_cols)} (cardinality ≥ {cardinality_threshold})")
        
        # Binary encoding (label encode to 0/1)
        binary_mappings = {}
        for col in binary_cols:
            categories = sorted(X_proc[col].dropna().unique())
            if len(categories) == 2:
                mapping = {categories[0]: 0, categories[1]: 1}
            else:
                mapping = {categories[0]: 0}
            
            binary_mappings[col] = mapping
            X_proc[col] = X_proc[col].map(mapping)
        
        # One-hot encoding
        onehot_columns = []
        for col in low_card_cols:
            dummies = pd.get_dummies(X_proc[col], prefix=col, drop_first=True, dtype=int)
            onehot_columns.extend(dummies.columns.tolist())
            X_proc = pd.concat([X_proc.drop(col, axis=1), dummies], axis=1)
        
        # Target encoding with KFold
        target_encoders = {}
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        
        for col in high_card_cols:            
            encoded_values = np.zeros(len(X_proc))
            
            for train_idx, val_idx in kf.split(X_proc):
                train_means = y.iloc[train_idx].groupby(X_proc[col].iloc[train_idx]).mean()
                encoded_values[val_idx] = X_proc[col].iloc[val_idx].map(train_means)
            
            global_mean = y.mean()
            encoded_values = pd.Series(encoded_values, index=X_proc.index).fillna(global_mean)
            
            full_mean_map = y.groupby(X_proc[col]).mean().to_dict()
            target_encoders[col] = {'mean_map': full_mean_map, 'global_mean': global_mean}
            X_proc[col] = encoded_values
        
        # Scaling
        scaler_dict = {
            'maxabs': MaxAbsScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler(),
            'minmax': MinMaxScaler()
        }
        scaler = scaler_dict[scaler_type]

        # Avoid scaling Binary/One-Hot columns to preserve sparsity and 0/1 logic.
        cols_to_scale = num_cols + high_card_cols

        # Ensure we only scale columns that exist in the processed dataframe
        cols_to_scale = [c for c in cols_to_scale if c in X_proc.columns]
        
        if cols_to_scale:
            X_proc[cols_to_scale] = scaler.fit_transform(X_proc[cols_to_scale])
        
        print(f"Scaling: {scaler_type.upper()} applied to {len(cols_to_scale)} features.")
        
        fitted_objects = {
            'scaler': scaler,
            'binary_mappings': binary_mappings,
            'target_encoders': target_encoders,
            'onehot_columns': onehot_columns,
            'cols_to_scale': cols_to_scale,
            'feature_names': list(X_proc.columns),
            'categorical_cols': cat_cols,
            'numerical_cols': num_cols,
            'binary_cols': binary_cols,
            'low_card_cols': low_card_cols,
            'high_card_cols': high_card_cols
        }
        
        return X_proc, fitted_objects
    
    else:
        # TEST MODE
        scaler = fitted_objects['scaler']
        binary_mappings = fitted_objects['binary_mappings']
        target_encoders = fitted_objects['target_encoders']
        onehot_columns = fitted_objects['onehot_columns']
        cols_to_scale = fitted_objects.get('cols_to_scale', [])
        binary_cols = fitted_objects['binary_cols']
        low_card_cols = fitted_objects['low_card_cols']
        high_card_cols = fitted_objects['high_card_cols']
        
        # Binary encoding
        for col in binary_cols:
            if col in X_proc.columns:
                mapping = binary_mappings[col]
                X_proc[col] = X_proc[col].map(mapping).fillna(0)
        
        # One-hot encoding
        for col in low_card_cols:
            if col in X_proc.columns:
                dummies_test = pd.get_dummies(X_proc[col], prefix=col, drop_first=True, dtype=int)
                
                expected_cols = [c for c in onehot_columns if c.startswith(col + '_')]

                dummies_aligned = dummies_test.reindex(columns=expected_cols, fill_value=0)
                X_proc = pd.concat([X_proc.drop(col, axis=1), dummies_aligned], axis=1)
        
        # Target encoding
        for col in high_card_cols:
            if col in X_proc.columns:
                mean_map = target_encoders[col]['mean_map']
                global_mean = target_encoders[col]['global_mean']
                X_proc[col] = X_proc[col].map(mean_map).fillna(global_mean)
        
        # Scaling
        # Check if columns to scale exist in test set
        test_cols_to_scale = [c for c in cols_to_scale if c in X_proc.columns]

        if test_cols_to_scale:
            X_proc[test_cols_to_scale] = scaler.transform(X_proc[test_cols_to_scale])

        # Final Alignment
        # Ensure test set has exactly the same columns in the same order as training
        # Missing columns (if any) are filled with 0
        X_proc = X_proc.reindex(columns=fitted_objects['feature_names'], fill_value=0)
        
        print(f"Test mode: {X_proc.shape}")
        
        return X_proc

In [ ]:
# Separate Features and Target for Training
X_final_train = train_df_final.drop("SalePrice", axis=1)
y_final_train = train_df_final["SalePrice"]

# Validation features are already separated in valid_df_final
X_final_valid = valid_df_final

# Fit and Transform Training Data
print("\n[Phase 1] Processing Training Data...")
X_train_proc, fitted_objects = encode_and_scale(
    X_final_train, 
    y=y_final_train,
    cardinality_threshold=20,
    n_splits=5,
    scaler_type='maxabs',
    random_state=42
)

# Fit and Transform Validation Data
# The function will use 'fitted_objects' to treat columns exactly as it did in training.
print("\n[Phase 2] Processing Validation Data...")
X_valid_proc = encode_and_scale(
    X_final_valid,
    fitted_objects=fitted_objects 
)

print(f"\nFinal Processed Shapes:")
print(f"X_train_proc: {X_train_proc.shape}")
print(f"X_val_proc:   {X_valid_proc.shape}")

In [ ]:
def train_and_optimize_models(
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    models='all',
    cv=5,
    n_trials=100,
    random_state=42,
    n_jobs=-1,
    verbose=1
):
    """
    Trains and optimizes multiple regression models using Optuna.
    Pruning: Stops unpromising trials early (saves RAM/time) for SVR, RF, XGB, LGBM, and CatBoost

    Parameters
    ----------
    X_train : pd.DataFrame or np.ndarray
        Training features (already encoded and scaled)
    y_train : pd.Series or np.ndarray
        Training target variable
    X_val : pd.DataFrame or np.ndarray, optional
        Validation features (for final evaluation)
    y_val : pd.Series or np.ndarray, optional
        Validation target variable
    models : str or list, default='all'
        Which models to train:
        - 'all': Train all 7 models
        - list: ['elasticnet', 'xgboost', 'lightgbm', ...]
    cv : int, default=5
        Number of cross-validation folds
    n_trials : int, default=100
        Number of Optuna trials per model
    random_state : int, default=42
        Random state for reproducibility
    n_jobs : int, default=-1
        Number of parallel jobs (-1 = use all cores)
    verbose : int, default=1
        Verbosity level (0=silent, 1=progress, 2=detailed)
    
    Returns
    -------
    results : dict
        Detailed results for each model with best_model, best_params, scores
    results_df : pd.DataFrame
        Summary DataFrame sorted by CV score
    
    Examples
    --------
    >>> # Train all models
    >>> results, df = train_and_optimize_models(X_train_proc, y_train, X_val_proc, y_val)
    >>> 
    >>> # Train only specific models (faster)
    >>> results, df = train_and_optimize_models(
    ...     X_train_proc, y_train, 
    ...     models=['elasticnet', 'xgboost', 'catboost'],
    ...     n_trials=50
    ... )
    """
    
    # Setup cross-validation
    kfold = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    
    # Suppress Optuna logs if verbose=0
    optuna.logging.set_verbosity(optuna.logging.WARNING if verbose == 0 else optuna.logging.INFO)
    
    # Objective Functions
    # Fast Models (Use cross_val_score, pruning is less critical here)
    def objective_elasticnet(trial):
        params = {
            'alpha': trial.suggest_float('alpha', 0.001, 10.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 0.9),
            'max_iter': trial.suggest_categorical('max_iter', [1000, 2000, 5000, 10000]),
            'random_state': random_state
        }
        model = ElasticNet(**params)
        scores = cross_val_score(model, X_train, y_train, cv=kfold, 
                                 scoring='neg_mean_squared_log_error', n_jobs=n_jobs)
        return np.sqrt(-scores.mean())
    
    def objective_knn(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 15),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan']),
            'n_jobs': n_jobs
        }
        model = KNeighborsRegressor(**params)
        scores = cross_val_score(model, X_train, y_train, cv=kfold,
                                 scoring='neg_mean_squared_log_error', n_jobs=n_jobs)
        return np.sqrt(-scores.mean())
    
    # Heavy Models (Manual CV loop for Pruning support)
    def objective_svr(trial):
        params = {
            'C': trial.suggest_float('C', 0.1, 100.0, log=True),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly']),
            'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            'epsilon': trial.suggest_float('epsilon', 0.01, 0.2)
        }
        
        fold_scores = []
        # Manual CV loop to enable pruning
        for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
            X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = SVR(**params)
            model.fit(X_tr, y_tr)
            
            # Predict and Clip (safety for RMSLE)
            y_pred = np.clip(model.predict(X_val_fold), 0, None)
            rmsle = np.sqrt(mean_squared_log_error(y_val_fold, y_pred))
            fold_scores.append(rmsle)
            
            # Report intermediate result to Optuna
            trial.report(rmsle, fold_idx)
            
            # Prune if the trial is unpromising
            if trial.should_prune():
                raise optuna.TrialPruned()
                
        return np.mean(fold_scores)
    
    def objective_random_forest(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 10, 30, step=5),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'random_state': random_state,
            'n_jobs': n_jobs 
        }
        
        fold_scores = []
        for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
            X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = RandomForestRegressor(**params)
            model.fit(X_tr, y_tr)
            
            y_pred = np.clip(model.predict(X_val_fold), 0, None)
            rmsle = np.sqrt(mean_squared_log_error(y_val_fold, y_pred))
            fold_scores.append(rmsle)
            
            trial.report(rmsle, fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
                
        return np.mean(fold_scores)
    
    def objective_xgboost(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'random_state': random_state,
            'n_jobs': n_jobs,
            'verbosity': 0
        }
        
        fold_scores = []
        for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
            X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = XGBRegressor(**params)
            model.fit(X_tr, y_tr)
            y_pred = np.clip(model.predict(X_val_fold), 0, None)
            rmsle = np.sqrt(mean_squared_log_error(y_val_fold, y_pred))
            fold_scores.append(rmsle)
            
            trial.report(rmsle, fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return np.mean(fold_scores)
    
    def objective_lightgbm(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 100),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'random_state': random_state,
            'n_jobs': n_jobs,
            'verbose': -1
        }
        
        fold_scores = []
        for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
            X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = LGBMRegressor(**params)
            model.fit(X_tr, y_tr)
            y_pred = np.clip(model.predict(X_val_fold), 0, None)
            rmsle = np.sqrt(mean_squared_log_error(y_val_fold, y_pred))
            fold_scores.append(rmsle)
            
            trial.report(rmsle, fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return np.mean(fold_scores)
    
    def objective_catboost(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 100, 300, step=50),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 7),
            'random_state': random_state,
            'verbose': 0
        }

        fold_scores = []
        for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
            X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = CatBoostRegressor(**params)
            model.fit(X_tr, y_tr)
            y_pred = np.clip(model.predict(X_val_fold), 0, None)
            rmsle = np.sqrt(mean_squared_log_error(y_val_fold, y_pred))
            fold_scores.append(rmsle)
            
            trial.report(rmsle, fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return np.mean(fold_scores)
    
    # Model registry
    MODEL_REGISTRY = {
        'elasticnet': objective_elasticnet,
        'knn': objective_knn,
        'svr': objective_svr,
        'random_forest': objective_random_forest,
        'xgboost': objective_xgboost,
        'lightgbm': objective_lightgbm,
        'catboost': objective_catboost
    }
    
    # Determine which models to train
    if models == 'all':
        selected_models = list(MODEL_REGISTRY.keys())
    else:
        selected_models = models
    
    # Validate model names
    invalid_models = set(selected_models) - set(MODEL_REGISTRY.keys())
    if invalid_models:
        raise ValueError(f"Invalid model names: {invalid_models}. Valid: {list(MODEL_REGISTRY.keys())}")
    

    # Pruner configuration
    # n_warmup_steps=2: Pruning starts after the 2nd fold (0 and 1 are safe).
    # This is a conservative approach to avoid killing potentially good models too early.
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=2)
    
    # Results storage
    results = {}
    summary_data = []
    
    if verbose >= 1:
        print(f"\n{'='*70}")
        print(f"Training {len(selected_models)} models with Optuna (RMSLE metric)")
        print(f"Train set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
        print(f"Trials per model: {n_trials} | CV folds: {cv}")
        if X_val is not None:
            print(f"Validation set: {X_val.shape[0]} samples")
        print(f"{'='*70}\n")
    
    # Train each model
    for i, model_name in enumerate(selected_models, 1):
        if verbose >= 1:
            print(f"[{i}/{len(selected_models)}] Optimizing {model_name.upper()} ({n_trials} trials)...")
        
        start_time = time.time()

        try:
            # Create study for each model
            study = optuna.create_study(
                direction='minimize',
                pruner=pruner,
                sampler=optuna.samplers.TPESampler(seed=random_state)
            )
            
            objective_func = MODEL_REGISTRY[model_name]
            
            # Run optimization
            # n_jobs=1: Using sequential optimization to save RAM and allow models 
            # to use internal parallelization (n_jobs=-1).
            study.optimize(
                objective_func,
                n_trials=n_trials,
                show_progress_bar=(verbose >= 1),
                n_jobs=1,
                catch=(Exception,) # Catches model-specific errors without crashing the loop
            )
            
            # Best params and score
            best_params = study.best_params
            cv_score = study.best_value
            
            # Re-instantiate the best model
            if model_name == 'elasticnet':
                best_model = ElasticNet(**best_params)
            elif model_name == 'knn':
                best_model = KNeighborsRegressor(**best_params, n_jobs=n_jobs)
            elif model_name == 'svr':
                best_model = SVR(**best_params)
            elif model_name == 'random_forest':
                best_model = RandomForestRegressor(**best_params, n_jobs=n_jobs)
            elif model_name == 'xgboost':
                best_model = XGBRegressor(**best_params)
            elif model_name == 'lightgbm':
                best_model = LGBMRegressor(**best_params)
            elif model_name == 'catboost':
                best_model = CatBoostRegressor(**best_params)
            
            # Fit on full training set
            best_model.fit(X_train, y_train)
            
            # Metrics on Train
            y_train_pred = np.clip(best_model.predict(X_train), 0, None)
            train_rmsle = np.sqrt(mean_squared_log_error(y_train, y_train_pred))
            train_mae = mean_absolute_error(y_train, y_train_pred)
            train_r2 = r2_score(y_train, y_train_pred)
            
            # Metrics on Validation (if provided)
            val_rmsle = val_mae = val_r2 = None
            if X_val is not None and y_val is not None:
                y_val_pred = np.clip(best_model.predict(X_val), 0, None)
                val_rmsle = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
                val_mae = mean_absolute_error(y_val, y_val_pred)
                val_r2 = r2_score(y_val, y_val_pred)
            
            elapsed = time.time() - start_time
            
            # Store results
            results[model_name] = {
                'best_model': best_model,
                'best_params': best_params,
                'cv_rmsle': cv_score,
                'train_rmsle': train_rmsle,
                'train_mae': train_mae,
                'train_r2': train_r2,
                'val_rmsle': val_rmsle,
                'val_mae': val_mae,
                'val_r2': val_r2,
                'fit_time': elapsed,
                'study': study
            }
            
            summary_data.append({
                'Model': model_name,
                'CV_RMSLE': cv_score,
                'Train_RMSLE': train_rmsle,
                'Train_R2': train_r2,
                'Train_MAE': train_mae,
                'Val_RMSLE': val_rmsle,
                'Val_R2': val_r2,
                'Val_MAE': val_mae,
                'Time(s)': elapsed
            })
        
            
            if verbose >= 1:
                print(f"--- {model_name.upper()} RESULTS ---")
                print(f"CV RMSLE: {cv_score:.4f}")
                print(f"Train RMSLE: {train_rmsle:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}")
                if val_rmsle: print(f"Val RMSLE: {val_rmsle:.4f} | MAE: {val_mae:.4f} | R2: {val_r2:.4f}")
                print(f"Time: {elapsed:.2f}s | Trials: {len(study.trials)}")
                print(f"Best params: {best_params}")
                print()
                
        except Exception as e:
            print(f"!! ERROR optimizing {model_name}: {str(e)}")
            continue

    # Final summary
    results_df = pd.DataFrame(summary_data).sort_values('CV_RMSLE').reset_index(drop=True)
    
    if verbose >= 1:
        print(f"{'='*70}")
        print("RESULTS SUMMARY (sorted by CV_RMSLE):")
        print(results_df.to_string(index=False))
        print(f"{'='*70}\n")
    
    return results, results_df

In [ ]:
# We use the y_valid we stored way back in the beginning (from the initial split)
# Ensure y_valid aligns with X_valid_proc dimensions (it should, 292 rows)
print("Starting Hyperparameter Optimization...")

optuna_results, summary_df = train_and_optimize_models(
    X_train=X_train_proc,
    y_train=y_final_train,
    X_val=X_valid_proc,
    y_val=y_valid,           # The original y_valid from the very first train_test_split
    models='all',          # Or select specific models: ['xgboost', 'lightgbm']
    cv=5,
    n_trials=200,          # Adjust based on your time constraints
    random_state=42,
    verbose=2
)

# Display the final leaderboard
print("\nFinal Model Leaderboard:")
display(summary_df)

In [ ]:
def train_dynamic_stacking(
    results,
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    included_models='all',
    meta_learner=None,
    cv=5,
    random_state=42,
    verbose=1
):
    """
    Trains and optimizes a Stacking Regressor using pre-tuned base models.
    
    This function dynamically reconstructs model instances using the 'best_params' 
    found in the previous optimization step. It avoids data leakage by retraining 
    base models from scratch within the stacking cross-validation scheme.
    
    It also automatically wraps boosting models (CatBoost, XGBoost, LightGBM)
    to prevent compatibility issues with newer Scikit-learn versions.

    Parameters
    ----------
    results : dict
        Dictionary containing results from 'train_and_optimize_models', specifically 
        requiring the 'best_params' key for each model.
    X_train : pd.DataFrame or np.ndarray
        Training features.
    y_train : pd.Series or np.ndarray
        Training target variable.
    X_val : pd.DataFrame or np.ndarray, optional
        Validation features for final evaluation.
    y_val : pd.Series or np.ndarray, optional
        Validation target variable.
    included_models : list or str, default='all'
        Specifies which models to include in the stack:
        - 'all': Includes all models present in the 'results' dictionary.
        - list: A specific list of model names, e.g., ['catboost', 'elasticnet'].
    meta_learner : sklearn estimator, optional
        The final estimator used to combine base model predictions.
        If None, defaults to RidgeCV (L2 regularised linear regression).
    cv : int, default=5
        Number of cross-validation folds for the stacking procedure.
    random_state : int, default=42
        Random state for reproducibility.
    verbose : int, default=1
        Verbosity level (0=silent, 1=progress/summary).

    Returns
    -------
    stacking_model : sklearn.ensemble.StackingRegressor
        The fitted Stacking Regressor ready for prediction.
    metrics : dict
        Performance metrics (RMSLE, MAE, R2) on training and validation sets.

    Examples
    --------
    >>> # Stack specific models from previous results
    >>> stack_model, metrics = train_dynamic_stacking(
    ...     results=results,
    ...     X_train=X_train, y_train=y_train,
    ...     included_models=['catboost', 'xgboost', 'elasticnet']
    ... )
    """
    
    # Model Factory: Maps string identifiers to class objects
    MODEL_FACTORY = {
        'xgboost': XGBRegressor,
        'lightgbm': LGBMRegressor,
        'catboost': CatBoostRegressor,
        'random_forest': RandomForestRegressor,
        'elasticnet': ElasticNet,
        'knn': KNeighborsRegressor,
        'svr': SVR
    }
    
    # Identify models to include in the stack
    available_models = list(results.keys())
    if included_models == 'all':
        target_models = available_models
    else:
        # Filter models: must exist in both 'results' and user request
        target_models = [m for m in included_models if m in available_models]
    
    if verbose >= 1:
        print(f"{'='*70}")
        print(f"Building Stacking Regressor with {len(target_models)} models")
        print(f"Included: {', '.join([m.upper() for m in target_models])}")
        print(f"{'='*70}")

    # Instantiate base estimators with optimized parameters
    estimators = []
    for model_name in target_models:
        if model_name not in MODEL_FACTORY:
            if verbose >= 1:
                print(f"Warning: No class definition found for '{model_name}'. Skipped.")
            continue
            
        best_params = results[model_name]['best_params']
        model_class = MODEL_FACTORY[model_name]
        
        # Handle random_state safety for applicable models
        if 'random_state' in best_params:
            model_instance = model_class(**best_params)
        elif hasattr(model_class(), 'random_state'):
            model_instance = model_class(**best_params, random_state=random_state)
        else:
            model_instance = model_class(**best_params)

        # Apply wrapper to specific boosting models to fix sklearn 1.6+ compatibility
        if model_name in ['catboost', 'xgboost', 'lightgbm']:
            model_instance = SklearnWrapper(model_instance)

        estimators.append((model_name, model_instance))
        
    # Configure Meta-Learner (Default: RidgeCV)
    if meta_learner is None:
        meta_learner = RidgeCV(alphas=[0.1, 1.0, 10.0])

    # Initialize and fit Stacking Regressor
    # n_jobs=-1 ensures parallel execution for base model training
    stacking_regressor = StackingRegressor(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=cv,
        n_jobs=-1,
        passthrough=False
    )
    
    if verbose >= 1:
        print("\nTraining Stacking Regressor (this may take time)...")
    
    stacking_regressor.fit(X_train, y_train)
    
    # Calculate performance metrics
    metrics = {}
    
    # Training metrics
    y_train_pred = np.clip(stacking_regressor.predict(X_train), 0, None)
    metrics['train_rmsle'] = np.sqrt(mean_squared_log_error(y_train, y_train_pred))
    metrics['train_mae'] = mean_absolute_error(y_train, y_train_pred)
    metrics['train_r2'] = r2_score(y_train, y_train_pred)
    
    # Validation metrics
    if X_val is not None and y_val is not None:
        y_val_pred = np.clip(stacking_regressor.predict(X_val), 0, None)
        metrics['val_rmsle'] = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
        metrics['val_mae'] = mean_absolute_error(y_val, y_val_pred)
        metrics['val_r2'] = r2_score(y_val, y_val_pred)
        
        if verbose >= 1:
            print(f"\nStacking Results:")
            print(f"Train RMSLE: {metrics['train_rmsle']:.4f}")
            print(f"Val RMSLE: {metrics['val_rmsle']:.4f}")
            print(f"Val R2   : {metrics['val_r2']:.4f}")
            print(f"{'='*70}\n")
    
    return stacking_regressor, metrics

In [ ]:
stack_model, stack_metrics = train_dynamic_stacking(
    optuna_results,
    X_train_proc,
    y_train,
    X_valid_proc,
    y_valid,
    included_models=["catboost", "xgboost", "lightgbm", "random_forest", "elasticnet"],
    cv=5,
    verbose=2
    )

# Let's see the final result
print("\n" + "="*50)
print("FINAL ENSEMBLE PERFORMANCE")
print("="*50)
print(f"Validation RMSLE: {stack_metrics['val_rmsle']:.5f}")
print(f"Validation R2   : {stack_metrics['val_r2']:.5f}")

### Production Build

In [ ]:
# ==============================================================================
# FINAL PRODUCTION BUILD
# ==============================================================================
# In this final section, we discard the ad-hoc preprocessing steps used during EDA.
# Instead, we utilize the robust 'HousePricePreprocessor' class defined in 'src.model_utils'.
# This ensures that the training data and production data are processed identically.

# 1. Initialize the Preprocessing Pipeline
# ------------------------------------------------------------------------------
print("Initializing production preprocessor...")
# The preprocessor contains all logic for cleaning, feature engineering, and scaling.
preprocessor = HousePricePreprocessor()

# Fit the preprocessor on the raw training data.
# CRITICAL: We use 'X_train' and 'y_train' from the initial split (Cell 7),
# which contain raw data before any notebook-specific transformations.
# This allows the class to learn medians, target encodings, and scaling parameters safely.
preprocessor.fit(X_train, y_train)

# Transform the datasets using the learned pipeline.
# This generates the exact feature set expected by the final model.
X_train_final = preprocessor.transform(X_train)
X_valid_final = preprocessor.transform(X_valid) # Note: Using 'X_valid' defined in split

print(f"Data processed successfully. Final shape: {X_train_final.shape}")

# 2. Train the Final Stacking Model
# ------------------------------------------------------------------------------
# We retrain the stacking model using the strictly processed data (X_train_final).
# This guarantees alignment between the preprocessor's output and the model's input.
# We reuse the 'results' dictionary from the optimization step to get the best hyperparameters.
print("Training final stacking model...")

final_stack_model, final_metrics = train_dynamic_stacking(
    results=optuna_results, 
    X_train=X_train_final, 
    y_train=y_train,
    X_val=X_valid_final, # Using the transformed validation set
    y_val=y_valid,       # Using 'y_valid' from the initial split
    # Only including the best performing models selected from our analysis
    included_models=['catboost', 'xgboost', 'elasticnet', 'lightgbm', 'random_forest'],
    verbose=2
)

# 3. Serialise and Save Artefacts
# ------------------------------------------------------------------------------
# We verify the output directory exists (handled by the save function) and save both components.

# A. Save the Preprocessor (The "Recipe")
# This object is needed to transform raw input data in production.
import joblib
preprocessor_path = "../models/preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)
print(f"Preprocessor saved to: {preprocessor_path}")

# B. Save the Model (The "Brain")
# Utilising our custom save function which attaches metadata.
model_path = save_production_model(
    final_stack_model, 
    final_metrics, 
    filename="final_model.joblib", 
    output_dir="../models"
)

# 4. Final Sanity Check
# ------------------------------------------------------------------------------
# Verify that the saved artefacts can be loaded and produce identical predictions.
# This simulates a real production inference call.
print("\n--- Performing Final Sanity Check ---")

loaded_model, _ = load_production_model(model_path)
loaded_preprocessor = joblib.load(preprocessor_path)

# Take one raw sample from the validation set
sample_raw_data = X_valid.iloc[:1] 

# Process it using the loaded pipeline (Simulation of API request)
processed_sample = loaded_preprocessor.transform(sample_raw_data) 

# Make a prediction
prediction = loaded_model.predict(processed_sample)

print(f"Raw Input Shape: {sample_raw_data.shape}")
print(f"Processed Shape: {processed_sample.shape}")
print(f"Prediction Value: {prediction[0]:.4f}")
print("Final pipeline build successful. Ready for deployment.")

## Submission Cell

In [ ]:
import joblib
from src.model_utils import load_production_model

# STEP 0: Load Saved Model & Preprocessor
print("="*70)
print("GENERATING SUBMISSION FOR TEST DATASET")
print("="*70)

print("\n[0/3] Loading Saved Model & Preprocessor...")
model_path = "../models/final_model.joblib"
preprocessor_path = "../models/preprocessor.joblib"

joblib_model, metadata = load_production_model(model_path)
preprocessor = joblib.load(preprocessor_path)
print(f"Model loaded from: {model_path}")
print(f"Preprocessor loaded from: {preprocessor_path}")

# STEP 1: Preprocessing (All-in-One)
print("\n[1/3] Preprocessing Test Data...")
# Preprocessor handles: Missing values, Feature Engineering, Encoding & Scaling
X_test_processed = preprocessor.transform(test_df)
print(f"Test data processed: {X_test_processed.shape}")

# STEP 2: Prediction
print("\n[2/3] Making Predictions...")
y_pred = joblib_model.predict(X_test_processed)
print(f"Predictions made: {len(y_pred)} predictions")
print(f"Range: ${y_pred.min():,.2f} - ${y_pred.max():,.2f}")
print(f"Mean:  ${y_pred.mean():,.2f}")

# STEP 3: Create Submission
print("\n[3/3] Creating Submission File...")
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': y_pred
})

submission.to_csv('../data/submission.csv', index=False)
print(f"Submission file saved: ../data/submission.csv")

print(f"\nFirst 10 predictions:")
display(submission.head(10))

print(f"\nLast 10 predictions:")
display(submission.tail(10))

print(f"\nStatistics:")
print(submission['SalePrice'].describe())

print("\n" + "="*70)
print("SUBMISSION READY!")
print("="*70)
